# AIC 2026 — InfoShot++ → Hugging Face `aic26-media` — TURBO v4 VIDEO — FAST keyframe uploader

Notebook Colab này tạo media projection cho frontend/backend hiện tại:

```text
SOURCE HF bucket
  Baonenha1/DATA-AIC-Keyframe/infoshootpp-v1/
  ├── keyframes/L21.tar ... L30.tar
  └── metadata/infoshootpp_metadata.tar

SOURCE video
  Cloudflare R2 /Videos/Videos_<group>/<video_id>.mp4

                      ↓ project + verify + cleanup

DESTINATION HF bucket (public)
  Baonenha1/aic26-media/
  ├── Keyframes/Keyframes_Lxx/<video_id>/<keyframe_n:03d>.jpg
  ├── Videos/Videos_Lxx/<video_id>.mp4
  └── manifest/...
```

Điểm quan trọng: ảnh nguồn là `f{frame_idx:08d}.jpg`, nhưng app hiện tại gọi
`{keyframe_n:03d}.jpg`. Notebook dùng `map-keyframes/*.csv` để tạo đúng alias;
không suy `keyframe_n == frame_idx`. Manifest vẫn giữ đầy đủ `frame_id`,
`frame_idx`, `pts_time`, `fps` để Milvus/Elastic/backend join chính xác.

Mỗi video/keyframe folder chỉ bị xóa local **sau khi** remote HF đã được kiểm
đúng path và byte size. Resume state nằm trên destination bucket, nên Colab bị
disconnect vẫn chạy tiếp được.

> **Ranh giới cutover:** notebook này chỉ migrate media, không reindex
> Milvus/Elasticsearch. Không đổi `MEDIA_BASE_URL` khi index còn dùng BTC
> ordinal. Index phải được rebuild theo InfoShot++ final `keyframe_n` (join
> canonical bằng `video_id + frame_idx`) trước; riêng L25 có hàng chục nghìn
> dòng `final n != btc_n` nên đổi base sớm sẽ trả sai ảnh hoặc 404.
> Destination này chỉ chứa L21–L30. Nếu live retrieval còn trả K01–K20, cần
> migrate/copy K media vào cùng destination hoặc scope index về L21–L30 trước,
> vì app chỉ có một global `MEDIA_BASE_URL`.


### FAST upload path

Bản này tối ưu riêng workload **~1.34M JPEG nhỏ**. Keyframe upload ưu tiên
Hugging Face Storage Bucket **S3 gateway** với nhiều concurrent `PutObject`
thay vì đưa JPG qua `batch_bucket_files`/Xet theo batch đồng bộ. JSON commit,
manifest và video vẫn dùng Hugging Face native API để giữ nguyên logic hiện tại.

Nếu `keyframe_upload_backend="s3"`, tạo S3 credentials từ dropdown của HF
Access Token và thêm hai Colab Secrets: `HF_S3_ACCESS_KEY_ID` (`HFAK...`) và
`HF_S3_SECRET_ACCESS_KEY`. Đây **không phải** chính chuỗi `hf_...`.


## Cách chạy

1. Mở **Runtime → Change runtime type**. Không cần GPU; chọn High-RAM nếu có.
2. Trong **Colab Secrets**, tạo `HF_TOKEN` và bật quyền truy cập notebook.
   Token cần read source bucket và write destination bucket.
   Để dùng fast S3 keyframe uploader (mặc định), trên trang HF Access Tokens mở
   dropdown của token → **Generate S3 credentials**, rồi thêm
   `HF_S3_ACCESS_KEY_ID` và `HF_S3_SECRET_ACCESS_KEY` vào Colab Secrets.
3. Kiểm tra cell `CONFIG`, rồi chạy tuần tự từ trên xuống.
4. Có thể chia category giữa nhiều Colab, nhưng không chạy hai notebook trên
   cùng một category. Với worker chỉ chạy keyframe, đặt `run_videos=False` và
   `run_finalize=False`. Sau khi mọi worker xong, dùng một runtime coordinator
   với `run_keyframes=False`, `run_videos=True`, `run_finalize=True`.
5. Chỉ sau khi Milvus/Elastic đã đồng bộ identity InfoShot++ final **và** không
   còn trả category ngoài L21–L30 (hoặc media đó đã được mirror), đặt backend:

```text
MEDIA_BASE_URL=https://huggingface.co/buckets/Baonenha1/aic26-media/resolve
```

Mặc định video dùng public R2 URL hiện có. Nếu muốn dùng S3 API riêng, đổi
`video_source_mode="r2_s3"` và thêm bốn Colab Secrets được ghi trong CONFIG.

Notebook không dùng `sync --delete` và không xóa bất kỳ object remote nào.
Hugging Face Storage Buckets không versioning; hãy dùng destination bucket mới
hoặc prefix mới khi thay đổi contract.


In [ ]:
%pip -q install "huggingface_hub[hf_xet]==1.24.0" "boto3>=1.35,<2" "pyarrow>=17,<24" "requests>=2.32,<3"


In [ ]:
# ===== USER CONFIG =====
CONFIG = {
    # Immutable InfoShot++ source.
    "source_bucket": "Baonenha1/DATA-AIC-Keyframe",
    "source_prefix": "infoshootpp-v1",

    # Recommended: a separate public media-only bucket. To reuse the source
    # bucket instead, set dest_bucket=source_bucket and dest_prefix="aic26-media".
    "dest_bucket": "Baonenha1/aic26-media",
    "dest_prefix": "",
    "create_dest_bucket_if_missing": True,
    "require_public_dest": True,

    # Distinct categories may be processed by distinct Colabs in parallel.
    "keyframe_categories": "L21,L22,L23,L24,L25,L26,L27,L28,L29,L30",
    # VIDEO TURBO v4 coordinator defaults:
    # keyframes are already migrated; only upload the 873 MP4s, then finalize.
    "run_keyframes": False,
    "run_videos": True,
    # Finalize chỉ chạy được khi destination đã có đủ 10 category + 873 video.
    "run_finalize": True,

    # auto: retain one TAR locally and extract only one video at a time when
    # disk permits; otherwise use Range streaming with per-video checkpoints.
    # local_tar: require enough disk. stream: never retain the full TAR.
    # local_tar recomputes full shard SHA-256. stream verifies declared SHA
    # binding + exact remote size/ranges/member inventory, not full TAR digest.
    "tar_mode": "auto",  # auto | local_tar | stream
    "disk_reserve_gib": 12.0,
    "max_video_stage_gib": 2.0,

    # Exact 873 L21-L30 videos are read from InfoShot++ run_summary.csv.
    "video_source_mode": "public_http",  # public_http | r2_s3
    "r2_public_base_url": "https://pub-c707387e6f0c4d4b9e37f00506a26f04.r2.dev",
    # r2_s3 secrets: R2_ACCOUNT_ID, R2_ACCESS_KEY_ID,
    # R2_SECRET_ACCESS_KEY, R2_BUCKET.

    # VIDEO TURBO v4 ---------------------------------------------------------
    # auto_bench uses real MP4s (no temporary objects) to compare native Xet
    # against the HF S3 gateway, then keeps the faster backend by MiB/s.
    "video_upload_backend": "auto_bench",  # auto_bench | xet | s3
    "video_benchmark_videos_per_backend": 2,

    # Bounded two-stage pipeline. Downloads and HF uploads overlap without
    # allowing the Colab disk to fill with the entire corpus.
    "video_download_workers": 4,
    "video_xet_upload_workers": 2,
    "video_s3_upload_workers": 4,
    "video_prefetch_limit": 8,

    # HF recommends very large multipart pieces on its S3 gateway so large
    # uploads use as few parts as possible.
    "video_s3_multipart_threshold_gib": 2.0,
    "video_s3_multipart_chunksize_gib": 2.0,
    "video_s3_part_workers": 4,

    # Video-mode Xet: keep adaptive concurrency enabled. This is deliberately
    # different from the old small-JPEG fixed-concurrency experiment.
    "video_xet_initial_upload_concurrency": 8,
    "video_xet_min_upload_concurrency": 2,
    "video_xet_max_upload_concurrency": 64,
    "video_xet_max_idle_connections": 64,
    "video_xet_file_ingestion_concurrency": 8,

    # If an MP4 exists remotely with the exact expected size but the runtime
    # died before writing its commit, re-download only to recover SHA-256 and
    # skip the HF upload.
    "video_resume_remote_size_match": True,

    "work_root": "/content/aic26_media_work",

    # TURBO keyframe uploader:
    # - auto_bench uploads DISJOINT real keyframes with Xet and S3 candidates,
    #   measures files/s, then keeps the fastest backend for this runtime.
    # - no benchmark bytes are wasted: benchmark files are part of the real video.
    "keyframe_upload_backend": "auto_bench",  # auto_bench | s3 | xet
    "hf_s3_workers": 192,                     # used when backend="s3"
    "hf_s3_worker_candidates": [64, 128, 192],
    "hf_s3_max_pool_connections": 256,
    "keyframe_benchmark_files_per_candidate": 192,

    # Mid-video crash resume: probe remote first, and if a partial upload is
    # detected, enumerate exact paths and skip files whose byte size already matches.
    "keyframe_partial_resume": True,
    "keyframe_partial_probe_paths": 16,

    # Xet tuning for many small JPEGs. HF documents 8 concurrent file ingestions
    # by default and 100 in high-performance mode. We raise only upload-side
    # controls instead of enabling the memory-heavy global HP flag.
    "xet_file_ingestion_concurrency": 100,
    "xet_fixed_upload_concurrency": 64,
    "xet_max_idle_connections": 128,

    "upload_batch_files": 1000,
    "verify_completed_on_resume": False,
    "max_retries": 8,
    "retry_base_seconds": 3.0,
    "http_connect_timeout": 30,
    "http_read_timeout": 300,

    # R2-compatible global CSV plus category Parquet files.
    "write_compat_media_manifest_csv": True,
    "manifest_parquet_batch_rows": 50_000,
    "strict_browser_checks": True,
    "cleanup_work_root_at_end": True,
    "disconnect_runtime_after_success": False,
    "run_self_tests": True,
}

PIPELINE_VERSION = "aic26-hf-media-projection-v1"
DEST_LAYOUT_VERSION = "r2-compatible-keyframe-n-alias-v1"
EXPECTED_VIDEOS = 873
EXPECTED_KEYFRAMES = 1_339_055
ALL_CATEGORIES = [f"L{i:02d}" for i in range(21, 31)]


In [ ]:
import csv
import hashlib
import json
import math
import os

# Must be set before importing huggingface_hub/hf_xet.
# VIDEO TURBO v4 uses adaptive Xet concurrency. HF documents adaptive mode as
# the default/recommended behavior and warns that the global HP flag is aimed at
# machines with >=64 GiB RAM, so we tune only the upload-side knobs on Colab.
if CONFIG.get("run_videos") and not CONFIG.get("run_keyframes"):
    os.environ["HF_XET_CLIENT_ENABLE_ADAPTIVE_CONCURRENCY"] = "true"
    os.environ.pop("HF_XET_FIXED_UPLOAD_CONCURRENCY", None)
    os.environ["HF_XET_CLIENT_AC_INITIAL_UPLOAD_CONCURRENCY"] = str(CONFIG["video_xet_initial_upload_concurrency"])
    os.environ["HF_XET_CLIENT_AC_MIN_UPLOAD_CONCURRENCY"] = str(CONFIG["video_xet_min_upload_concurrency"])
    os.environ["HF_XET_CLIENT_AC_MAX_UPLOAD_CONCURRENCY"] = str(CONFIG["video_xet_max_upload_concurrency"])
    os.environ["HF_XET_CLIENT_MAX_IDLE_CONNECTIONS"] = str(CONFIG["video_xet_max_idle_connections"])
    os.environ["HF_XET_DATA_MAX_CONCURRENT_FILE_INGESTION"] = str(CONFIG["video_xet_file_ingestion_concurrency"])
else:
    # Preserve v3 behavior if this notebook is deliberately repurposed for
    # small-JPEG keyframe migration.
    os.environ["HF_XET_DATA_MAX_CONCURRENT_FILE_INGESTION"] = str(CONFIG["xet_file_ingestion_concurrency"])
    os.environ["HF_XET_FIXED_UPLOAD_CONCURRENCY"] = str(CONFIG["xet_fixed_upload_concurrency"])
    os.environ["HF_XET_CLIENT_MAX_IDLE_CONNECTIONS"] = str(CONFIG["xet_max_idle_connections"])

import random
import re
import shutil
import tarfile
import time
import threading
from collections import deque
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath
from typing import Any, Callable, Iterable, Sequence
from urllib.parse import quote

import boto3
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import requests
from botocore.config import Config as BotoConfig
from boto3.s3.transfer import TransferConfig
from huggingface_hub import (
    BucketFile,
    HfApi,
    HfFileSystem,
    batch_bucket_files,
    bucket_info,
    create_bucket,
)
from huggingface_hub.errors import BucketNotFoundError
from tqdm.auto import tqdm

try:
    from google.colab import userdata
except ImportError:  # lets the notebook helpers be inspected outside Colab
    userdata = None


def read_secret(name: str, *, required: bool = True) -> str | None:
    value = None
    if userdata is not None:
        try:
            value = userdata.get(name)
        except Exception:
            value = None
    if not value:
        value = os.environ.get(name)
    value = value.strip() if value else None
    if required and not value:
        raise RuntimeError(f"Thiếu Colab Secret: {name}")
    return value


HF_TOKEN = read_secret("HF_TOKEN")
assert HF_TOKEN and HF_TOKEN.startswith("hf_"), "HF_TOKEN không đúng định dạng"

# S3 credentials are generated from the HF token's dropdown in Access Tokens.
# They are AWS-style HFAK... credentials, not the hf_... token itself.
HF_S3_ACCESS_KEY_ID = read_secret("HF_S3_ACCESS_KEY_ID", required=False)
HF_S3_SECRET_ACCESS_KEY = read_secret("HF_S3_SECRET_ACCESS_KEY", required=False)
if bool(HF_S3_ACCESS_KEY_ID) != bool(HF_S3_SECRET_ACCESS_KEY):
    raise RuntimeError("HF S3 credentials phải có đủ HF_S3_ACCESS_KEY_ID và HF_S3_SECRET_ACCESS_KEY")

SOURCE_BUCKET = str(CONFIG["source_bucket"]).strip().strip("/")
SOURCE_PREFIX = str(CONFIG["source_prefix"]).strip().strip("/")
DEST_BUCKET = str(CONFIG["dest_bucket"]).strip().strip("/")
DEST_PREFIX = str(CONFIG["dest_prefix"]).strip().strip("/")
WORK_ROOT = Path(CONFIG["work_root"]).resolve()

if "/" not in DEST_BUCKET:
    raise ValueError("dest_bucket phải có dạng namespace/bucket để dùng HF Storage Bucket")
DEST_NAMESPACE, DEST_S3_BUCKET = DEST_BUCKET.split("/", 1)
if not DEST_NAMESPACE or not DEST_S3_BUCKET or "/" in DEST_S3_BUCKET:
    raise ValueError(f"dest_bucket không hợp lệ cho HF S3 gateway: {DEST_BUCKET}")

KEYFRAME_UPLOAD_BACKEND = str(CONFIG["keyframe_upload_backend"]).strip().lower()
if KEYFRAME_UPLOAD_BACKEND == "auto":
    KEYFRAME_UPLOAD_BACKEND = "auto_bench"
if KEYFRAME_UPLOAD_BACKEND not in {"s3", "auto_bench", "xet"}:
    raise ValueError("keyframe_upload_backend phải là auto_bench, s3 hoặc xet")
if KEYFRAME_UPLOAD_BACKEND == "s3" and not HF_S3_ACCESS_KEY_ID:
    raise RuntimeError(
        "keyframe_upload_backend='s3' cần Colab Secrets HF_S3_ACCESS_KEY_ID + "
        "HF_S3_SECRET_ACCESS_KEY."
    )
if KEYFRAME_UPLOAD_BACKEND == "auto_bench" and not HF_S3_ACCESS_KEY_ID:
    print("[TURBO] Không có HF S3 credentials -> benchmark bị giới hạn, dùng Xet.")
    KEYFRAME_UPLOAD_BACKEND = "xet"

VIDEO_UPLOAD_BACKEND = str(CONFIG.get("video_upload_backend", "xet")).strip().lower()
if VIDEO_UPLOAD_BACKEND not in {"auto_bench", "xet", "s3"}:
    raise ValueError("video_upload_backend phải là auto_bench, xet hoặc s3")
if VIDEO_UPLOAD_BACKEND == "s3" and not HF_S3_ACCESS_KEY_ID:
    raise RuntimeError(
        "video_upload_backend='s3' cần HF_S3_ACCESS_KEY_ID + HF_S3_SECRET_ACCESS_KEY"
    )
if VIDEO_UPLOAD_BACKEND == "auto_bench" and not HF_S3_ACCESS_KEY_ID:
    print("[VIDEO TURBO] Không có HF S3 credentials -> dùng Xet.")
    VIDEO_UPLOAD_BACKEND = "xet"

if WORK_ROOT == Path("/content") or Path("/content") not in WORK_ROOT.parents:
    raise ValueError("work_root phải là thư mục con cụ thể của /content")
if CONFIG["tar_mode"] not in {"auto", "local_tar", "stream"}:
    raise ValueError("tar_mode phải là auto, local_tar hoặc stream")
if SOURCE_BUCKET == DEST_BUCKET:
    if not DEST_PREFIX:
        raise ValueError("Cùng bucket thì dest_prefix không được rỗng")
    if DEST_PREFIX == SOURCE_PREFIX or DEST_PREFIX.startswith(SOURCE_PREFIX + "/"):
        raise ValueError("Destination không được nằm trong source prefix")

REQUEST_HEADERS = {"Authorization": f"Bearer {HF_TOKEN}"}
HTTP = requests.Session()
API = HfApi(token=HF_TOKEN)
HFFS = HfFileSystem(token=HF_TOKEN)


def build_hf_s3_client():
    if not HF_S3_ACCESS_KEY_ID or not HF_S3_SECRET_ACCESS_KEY:
        return None
    workers = int(CONFIG["hf_s3_workers"])
    pool = max(workers, int(CONFIG["hf_s3_max_pool_connections"]))
    return boto3.client(
        "s3",
        endpoint_url=f"https://s3.hf.co/{DEST_NAMESPACE}",
        aws_access_key_id=HF_S3_ACCESS_KEY_ID,
        aws_secret_access_key=HF_S3_SECRET_ACCESS_KEY,
        region_name="us-east-1",
        config=BotoConfig(
            region_name="us-east-1",
            s3={"addressing_style": "path"},
            request_checksum_calculation="when_required",
            response_checksum_validation="when_required",
            max_pool_connections=pool,
            connect_timeout=int(CONFIG["http_connect_timeout"]),
            read_timeout=int(CONFIG["http_read_timeout"]),
            tcp_keepalive=True,
            retries={"max_attempts": 5, "mode": "adaptive"},
        ),
    )


HF_S3 = build_hf_s3_client()
print(
    "Keyframe uploader:",
    {
        "backend": KEYFRAME_UPLOAD_BACKEND,
        "s3_workers": (
            list(CONFIG["hf_s3_worker_candidates"])
            if KEYFRAME_UPLOAD_BACKEND == "auto_bench"
            else int(CONFIG["hf_s3_workers"]) if KEYFRAME_UPLOAD_BACKEND == "s3" else None
        ),
        "s3_pool": int(CONFIG["hf_s3_max_pool_connections"]) if HF_S3 is not None else None,
        "xet_file_ingestion": int(CONFIG["xet_file_ingestion_concurrency"]),
        "xet_fixed_upload": int(CONFIG["xet_fixed_upload_concurrency"]),
    },
)

print(
    "Video TURBO v4:",
    {
        "backend": VIDEO_UPLOAD_BACKEND,
        "download_workers": int(CONFIG["video_download_workers"]),
        "xet_upload_workers": int(CONFIG["video_xet_upload_workers"]),
        "s3_upload_workers": int(CONFIG["video_s3_upload_workers"]),
        "prefetch_limit": int(CONFIG["video_prefetch_limit"]),
        "xet_adaptive": bool(CONFIG.get("run_videos") and not CONFIG.get("run_keyframes")),
    },
)

for subdir in ["downloads", "metadata", "staging/keyframes", "staging/videos", "commits", "manifests"]:
    (WORK_ROOT / subdir).mkdir(parents=True, exist_ok=True)


def parse_categories(value: str | Sequence[str]) -> list[str]:
    if isinstance(value, str):
        tokens = re.split(r"[,;\s]+", value.strip()) if value.strip() else []
    else:
        tokens = [str(item).strip() for item in value]
    tokens = list(dict.fromkeys(token.upper() for token in tokens if token))
    unknown = sorted(set(tokens) - set(ALL_CATEGORIES))
    if unknown:
        raise ValueError(f"Category không hợp lệ: {unknown}")
    return [category for category in ALL_CATEGORIES if category in tokens]


SELECTED_CATEGORIES = parse_categories(CONFIG["keyframe_categories"])
if CONFIG["run_keyframes"] and not SELECTED_CATEGORIES:
    raise ValueError("run_keyframes=True nhưng keyframe_categories rỗng")
if (
    CONFIG["run_keyframes"]
    and CONFIG["run_videos"]
    and SELECTED_CATEGORIES != ALL_CATEGORIES
):
    raise ValueError(
        "Category worker không được chạy video queue dùng chung. "
        "Đặt run_videos=False và run_finalize=False; chỉ một coordinator chạy video."
    )

print("Pipeline:", PIPELINE_VERSION)
print("Source:", f"hf://buckets/{SOURCE_BUCKET}/{SOURCE_PREFIX}/")
print("Destination:", f"hf://buckets/{DEST_BUCKET}/{DEST_PREFIX}/" if DEST_PREFIX else f"hf://buckets/{DEST_BUCKET}/")
print("Categories this runtime:", SELECTED_CATEGORIES)
print("Work root:", WORK_ROOT)


## Preflight bucket và public URL

Cell này chỉ tạo destination bucket khi nó chưa tồn tại và CONFIG cho phép.
Nó không thay đổi visibility của bucket đã tồn tại. Nếu bucket đang private,
cell dừng để tránh upload hàng trăm GB mà frontend không thể đọc.


In [ ]:
source_info = bucket_info(SOURCE_BUCKET, token=HF_TOKEN)
print(
    "Source bucket:",
    {"id": source_info.id, "private": source_info.private, "files": source_info.total_files, "bytes": source_info.size},
)

try:
    dest_info = bucket_info(DEST_BUCKET, token=HF_TOKEN)
except BucketNotFoundError:
    if not CONFIG["create_dest_bucket_if_missing"]:
        raise
    print("Destination chưa tồn tại; đang tạo public bucket:", DEST_BUCKET)
    create_bucket(DEST_BUCKET, private=False, exist_ok=True, token=HF_TOKEN)
    dest_info = bucket_info(DEST_BUCKET, token=HF_TOKEN)

if CONFIG["require_public_dest"] and dest_info.private:
    raise RuntimeError(
        f"Destination {DEST_BUCKET} đang private. Hãy mở bucket Settings trên HF, "
        "chuyển sang public, rồi chạy lại cell này."
    )

def remote_join(*parts: str) -> str:
    return "/".join(str(part).strip("/") for part in parts if str(part).strip("/"))


def source_key(relative: str) -> str:
    return remote_join(SOURCE_PREFIX, relative)


def dest_key(relative: str) -> str:
    return remote_join(DEST_PREFIX, relative)


def resolve_url(bucket: str, path: str) -> str:
    return f"https://huggingface.co/buckets/{bucket}/resolve/{quote(path, safe='/')}"


def public_media_base_url() -> str:
    base = f"https://huggingface.co/buckets/{DEST_BUCKET}/resolve"
    return f"{base}/{quote(DEST_PREFIX, safe='/')}" if DEST_PREFIX else base


MEDIA_BASE_URL = public_media_base_url().rstrip("/")
print(
    "Destination bucket:",
    {"id": dest_info.id, "private": dest_info.private, "files": dest_info.total_files, "bytes": dest_info.size},
)
print("MEDIA_BASE_URL:", MEDIA_BASE_URL)


## Helper an toàn, retry, upload và remote verification

Mọi local delete đều bị giới hạn trong `work_root`. Native Xet upload được chunk tối đa
1.000 object/call. Keyframe FAST path dùng HF S3 gateway với concurrent `PutObject`,
tránh giới hạn ingest-file mặc định của Xet cho workload rất nhiều JPEG nhỏ. Một video chỉ được commit sau khi destination có đúng toàn bộ
path và size; object thừa dưới cùng video prefix làm audit fail thay vì bị xóa.


**TURBO v3:** keyframe upload có mid-video resume và empirical auto-benchmark giữa native Xet (100 file ingestion / fixed 64 upload streams) với HF S3 64/128/192 workers. Benchmark dùng chính các file thật, không tạo object tạm.


In [ ]:
GIB = 1024**3
MEMBER_RE = re.compile(
    r"^infoshootpp/keyframes/(?P<category>L\d{2})/(?P<video_id>L\d{2}_V\d+)/f(?P<frame_idx>\d{8})\.jpg$"
)


def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


def stable_json_bytes(value: Any) -> bytes:
    return (json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":")) + "\n").encode("utf-8")


def sha256_bytes(value: bytes) -> str:
    return hashlib.sha256(value).hexdigest()


def sha256_file(path: Path, *, chunk_size: int = 16 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def atomic_json(path: Path, value: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    partial = path.with_name(path.name + ".part")
    partial.write_bytes(stable_json_bytes(value))
    os.replace(partial, path)


def assert_in_work_root(path: Path) -> Path:
    resolved = path.resolve()
    if resolved == WORK_ROOT or WORK_ROOT not in resolved.parents:
        raise ValueError(f"Refuse operation outside work_root: {resolved}")
    return resolved


def remove_work_path(path: Path) -> None:
    resolved = assert_in_work_root(path)
    if resolved.is_dir():
        shutil.rmtree(resolved)
    elif resolved.exists():
        resolved.unlink()


def retry(label: str, operation: Callable[[], Any], *, max_retries: int | None = None) -> Any:
    limit = int(CONFIG["max_retries"] if max_retries is None else max_retries)
    base = float(CONFIG["retry_base_seconds"])
    for attempt in range(limit + 1):
        try:
            return operation()
        except KeyboardInterrupt:
            raise
        except Exception as exc:
            message = f"{type(exc).__name__}: {exc}"
            lowered = message.lower()
            if any(marker in lowered for marker in ["401", "403", "unauthorized", "forbidden", "invalid token"]):
                raise RuntimeError(f"Lỗi không retry được [{label}]: {message}") from exc
            if attempt >= limit:
                raise RuntimeError(f"Hết retry [{label}]: {message}") from exc
            delay = min(180.0, base * (2 ** min(attempt, 6))) + random.random()
            print(f"[retry {attempt + 1}/{limit}] {label}: {message}; chờ {delay:.1f}s", flush=True)
            time.sleep(delay)
    raise AssertionError("unreachable")


# Run this only after retry() exists. The previous FAST notebook placed this
# preflight in an earlier cell and therefore raised NameError on a clean run.
if KEYFRAME_UPLOAD_BACKEND in {"s3", "auto_bench"} and HF_S3 is not None:
    retry(
        "HF S3 gateway preflight",
        lambda: HF_S3.list_objects_v2(Bucket=DEST_S3_BUCKET, MaxKeys=1),
    )
    print(f"HF S3 gateway PASS: https://s3.hf.co/{DEST_NAMESPACE}/{DEST_S3_BUCKET}")


def get_bucket_file(bucket: str, path: str) -> BucketFile | None:
    items = list(API.get_bucket_paths_info(bucket, [path], token=HF_TOKEN))
    return next((item for item in items if item.path == path), None)


def read_remote_json(bucket: str, path: str) -> dict[str, Any] | None:
    """Read bucket JSON through the canonical resolve URL, bypassing fsspec cache."""
    url = resolve_url(bucket, path)

    def _read() -> dict[str, Any] | None:
        response = HTTP.get(
            url,
            headers=REQUEST_HEADERS,
            allow_redirects=True,
            timeout=(int(CONFIG["http_connect_timeout"]), int(CONFIG["http_read_timeout"])),
        )
        if response.status_code == 404:
            response.close()
            return None
        try:
            response.raise_for_status()
            value = response.json()
        finally:
            response.close()
        if not isinstance(value, dict):
            raise TypeError(f"Remote JSON không phải object: {path}")
        return value

    return retry(f"read {path}", _read)


def upload_pairs(pairs: Sequence[tuple[str | Path | bytes, str]], *, label: str) -> None:
    """Native HF/Xet uploader for JSON/manifests/videos."""
    batch_size = int(CONFIG["upload_batch_files"])
    if not 1 <= batch_size <= 1000:
        raise ValueError("upload_batch_files phải trong 1..1000")
    for offset in range(0, len(pairs), batch_size):
        chunk = list(pairs[offset : offset + batch_size])
        retry(
            f"{label} files {offset + 1}-{offset + len(chunk)}",
            lambda chunk=chunk: batch_bucket_files(DEST_BUCKET, add=chunk, token=HF_TOKEN),
        )


def _hf_s3_put_file(local_file: Path, remote_path: str) -> int:
    if HF_S3 is None:
        raise RuntimeError("HF S3 client chưa được cấu hình")
    size = int(local_file.stat().st_size)

    def _put() -> None:
        with local_file.open("rb") as body:
            HF_S3.put_object(
                Bucket=DEST_S3_BUCKET,
                Key=remote_path,
                Body=body,
                ContentLength=size,
                ContentType="image/jpeg",
            )

    retry(f"HF S3 PUT {remote_path}", _put, max_retries=3)
    return size


def _stats(backend: str, pairs: Sequence[tuple[Path, str]], started: float, *, workers: int | None = None) -> dict[str, Any]:
    elapsed = max(time.perf_counter() - started, 1e-9)
    total_bytes = sum(int(Path(local).stat().st_size) for local, _ in pairs)
    result = {
        "backend": backend,
        "workers": workers,
        "files": len(pairs),
        "bytes": total_bytes,
        "seconds": elapsed,
        "files_per_s": len(pairs) / elapsed,
        "mib_per_s": total_bytes / 2**20 / elapsed,
    }
    return result


def _print_upload_stats(label: str, stat: dict[str, Any]) -> None:
    worker_text = f" workers={stat['workers']}" if stat.get("workers") else ""
    print(
        f"[UPLOAD] {label}: {stat['files']:,} files, {stat['bytes'] / 2**20:.1f} MiB "
        f"in {stat['seconds']:.1f}s | {stat['files_per_s']:.1f} files/s | "
        f"{stat['mib_per_s']:.1f} MiB/s | backend={stat['backend']}{worker_text}",
        flush=True,
    )


def _upload_keyframes_xet(
    pairs: Sequence[tuple[Path, str]],
    *,
    label: str,
) -> dict[str, Any]:
    """Native Xet path: pass all files at once; huggingface_hub chunks at 1000."""
    normalized = [(Path(local), remote) for local, remote in pairs]
    if not normalized:
        return _stats("XET", normalized, time.perf_counter())
    started = time.perf_counter()
    # Calling batch_bucket_files once preserves one Xet progress reporter across
    # the library's internal 1000-file chunks.
    retry(
        label,
        lambda: batch_bucket_files(DEST_BUCKET, add=normalized, token=HF_TOKEN),
    )
    stat = _stats("XET", normalized, started)
    _print_upload_stats(label, stat)
    return stat


def _upload_keyframes_s3(
    pairs: Sequence[tuple[Path, str]],
    *,
    label: str,
    workers: int,
) -> dict[str, Any]:
    normalized = [(Path(local), remote) for local, remote in pairs]
    if not normalized:
        return _stats("S3", normalized, time.perf_counter(), workers=workers)
    if HF_S3 is None:
        raise RuntimeError("S3 keyframe backend được chọn nhưng HF S3 credentials chưa sẵn sàng")
    if not 1 <= workers <= 256:
        raise ValueError("S3 workers phải trong 1..256")

    started = time.perf_counter()
    completed_bytes = 0
    last_stats_at = started
    with ThreadPoolExecutor(max_workers=workers, thread_name_prefix="hf-s3") as executor:
        futures = {
            executor.submit(_hf_s3_put_file, local, remote): (local, remote)
            for local, remote in normalized
        }
        with tqdm(total=len(futures), desc=label, unit="file", leave=False) as bar:
            for future in as_completed(futures):
                completed_bytes += int(future.result())
                bar.update(1)
                now = time.perf_counter()
                if now - last_stats_at >= 0.5 or bar.n == len(futures):
                    elapsed = max(now - started, 1e-9)
                    bar.set_postfix(
                        {
                            "files/s": f"{bar.n / elapsed:.1f}",
                            "MiB/s": f"{completed_bytes / 2**20 / elapsed:.1f}",
                        },
                        refresh=False,
                    )
                    last_stats_at = now

    stat = _stats("S3", normalized, started, workers=workers)
    _print_upload_stats(label, stat)
    return stat


def _get_matching_remote_paths(
    pairs: Sequence[tuple[Path, str]],
    *,
    label: str,
) -> set[str]:
    """Return remote paths that already exist with the exact local byte size."""
    if not pairs or not CONFIG.get("keyframe_partial_resume", True):
        return set()

    normalized = [(Path(local), remote) for local, remote in pairs]
    expected_size = {remote: int(local.stat().st_size) for local, remote in normalized}
    paths = list(expected_size)

    # Cheap probe first. New videos pay one tiny metadata request; only an
    # interrupted/partial video triggers the full path scan.
    probe_n = max(1, int(CONFIG.get("keyframe_partial_probe_paths", 16)))
    if len(paths) <= probe_n:
        probe_paths = paths
    else:
        indices = sorted({
            round(i * (len(paths) - 1) / (probe_n - 1))
            for i in range(probe_n)
        }) if probe_n > 1 else [0]
        probe_paths = [paths[i] for i in indices]

    probe_items = retry(
        f"{label} partial-resume probe",
        lambda: list(API.get_bucket_paths_info(DEST_BUCKET, probe_paths, token=HF_TOKEN)),
    )
    if not probe_items:
        return set()

    print(f"[RESUME] {label}: partial remote data detected; scanning exact paths...", flush=True)
    matching: set[str] = set()
    for offset in range(0, len(paths), 500):
        chunk = paths[offset : offset + 500]
        items = retry(
            f"{label} resume paths {offset + 1}-{offset + len(chunk)}",
            lambda chunk=chunk: list(API.get_bucket_paths_info(DEST_BUCKET, chunk, token=HF_TOKEN)),
        )
        for item in items:
            if item.path in expected_size and int(item.size) == expected_size[item.path]:
                matching.add(item.path)
    return matching


_KEYFRAME_TUNED_BACKEND: str | None = None
_KEYFRAME_TUNED_S3_WORKERS: int | None = None


def _autotune_keyframe_backend(
    pairs: Sequence[tuple[Path, str]],
    *,
    label: str,
) -> list[tuple[Path, str]]:
    """Benchmark on disjoint REAL files; returns files not consumed by benchmark."""
    global _KEYFRAME_TUNED_BACKEND, _KEYFRAME_TUNED_S3_WORKERS

    if _KEYFRAME_TUNED_BACKEND is not None:
        return list(pairs)

    if HF_S3 is None:
        _KEYFRAME_TUNED_BACKEND = "xet"
        print("[TURBO] S3 unavailable -> selected XET", flush=True)
        return list(pairs)

    candidates = [int(x) for x in CONFIG.get("hf_s3_worker_candidates", [64, 128, 192])]
    candidates = [x for x in candidates if 1 <= x <= 256]
    specs: list[tuple[str, int | None]] = [("xet", None)] + [("s3", x) for x in candidates]

    requested = int(CONFIG.get("keyframe_benchmark_files_per_candidate", 192))
    per_candidate = min(requested, len(pairs) // len(specs))
    if per_candidate < 32:
        # Too few files for a meaningful benchmark. Prefer tuned Xet.
        _KEYFRAME_TUNED_BACKEND = "xet"
        print(
            f"[TURBO] only {len(pairs)} pending files; skip benchmark -> XET",
            flush=True,
        )
        return list(pairs)

    bench_total = per_candidate * len(specs)
    bench_pool = list(pairs[:bench_total])
    remaining = list(pairs[bench_total:])

    # Round-robin assignment balances JPEG-size distribution between candidates.
    groups: list[list[tuple[Path, str]]] = [[] for _ in specs]
    for index, pair in enumerate(bench_pool):
        groups[index % len(specs)].append(pair)

    results: list[dict[str, Any]] = []
    failed_pairs: list[tuple[Path, str]] = []
    print(
        f"[TURBO] empirical benchmark: {per_candidate} real files/candidate | "
        + ", ".join("XET" if b == "xet" else f"S3-{w}" for b, w in specs),
        flush=True,
    )

    for (backend, workers), group in zip(specs, groups):
        bench_label = f"{label} BENCH {'XET' if backend == 'xet' else f'S3-{workers}'}"
        try:
            if backend == "xet":
                stat = _upload_keyframes_xet(group, label=bench_label)
            else:
                assert workers is not None
                stat = _upload_keyframes_s3(group, label=bench_label, workers=workers)
            results.append(stat)
        except Exception as exc:
            print(
                f"[TURBO] candidate {backend}/{workers} failed: {type(exc).__name__}: {exc}",
                flush=True,
            )
            # Safe: re-uploading a partially successful group is idempotent.
            failed_pairs.extend(group)

    if not results:
        raise RuntimeError("Tất cả keyframe upload benchmark candidates đều thất bại")

    # Small JPEGs are dominated by per-object overhead, so files/s is the primary score.
    best = max(results, key=lambda x: (float(x["files_per_s"]), float(x["mib_per_s"])))
    _KEYFRAME_TUNED_BACKEND = str(best["backend"]).lower()
    _KEYFRAME_TUNED_S3_WORKERS = int(best["workers"]) if best.get("workers") else None

    print("[TURBO] benchmark results:", flush=True)
    for stat in sorted(results, key=lambda x: float(x["files_per_s"]), reverse=True):
        wt = f"-{stat['workers']}" if stat.get("workers") else ""
        print(
            f"  {stat['backend']}{wt}: {stat['files_per_s']:.1f} files/s | "
            f"{stat['mib_per_s']:.1f} MiB/s",
            flush=True,
        )
    selected = (
        "XET"
        if _KEYFRAME_TUNED_BACKEND == "xet"
        else f"S3-{_KEYFRAME_TUNED_S3_WORKERS}"
    )
    print(f"[TURBO] SELECTED for remaining runtime: {selected}", flush=True)

    return remaining + failed_pairs


def upload_keyframe_pairs(
    pairs: Sequence[tuple[Path, str]],
    *,
    label: str,
) -> None:
    """Upload many small JPEGs with mid-video resume and empirical backend tuning."""
    normalized = [(Path(local), remote) for local, remote in pairs]
    if not normalized:
        return

    # Resume an interrupted video without re-PUTing files that are already correct.
    matching = _get_matching_remote_paths(normalized, label=label)
    if matching:
        before = len(normalized)
        normalized = [(local, remote) for local, remote in normalized if remote not in matching]
        print(
            f"[RESUME] {label}: skip {before - len(normalized):,}/{before:,} already-correct files; "
            f"pending={len(normalized):,}",
            flush=True,
        )
        if not normalized:
            return

    global _KEYFRAME_TUNED_BACKEND, _KEYFRAME_TUNED_S3_WORKERS

    if KEYFRAME_UPLOAD_BACKEND == "xet":
        _upload_keyframes_xet(normalized, label=label)
        return

    if KEYFRAME_UPLOAD_BACKEND == "s3":
        _upload_keyframes_s3(
            normalized,
            label=label,
            workers=int(CONFIG["hf_s3_workers"]),
        )
        return

    # auto_bench: benchmark only once per runtime, on real disjoint files.
    if _KEYFRAME_TUNED_BACKEND is None:
        normalized = _autotune_keyframe_backend(normalized, label=label)
        if not normalized:
            return

    if _KEYFRAME_TUNED_BACKEND == "xet":
        _upload_keyframes_xet(normalized, label=label)
    elif _KEYFRAME_TUNED_BACKEND == "s3":
        _upload_keyframes_s3(
            normalized,
            label=label,
            workers=int(_KEYFRAME_TUNED_S3_WORKERS or CONFIG["hf_s3_workers"]),
        )
    else:
        raise RuntimeError(f"Unknown tuned backend: {_KEYFRAME_TUNED_BACKEND}")




def upload_json(path: str, value: Any, *, label: str) -> tuple[int, str]:
    payload = stable_json_bytes(value)
    upload_pairs([(payload, path)], label=label)
    remote = get_bucket_file(DEST_BUCKET, path)
    if remote is None or int(remote.size) != len(payload):
        raise RuntimeError(f"Remote JSON verify lỗi: {path}")
    return len(payload), sha256_bytes(payload)


def list_remote_files(bucket: str, prefix: str) -> dict[str, int]:
    files: dict[str, int] = {}
    for item in API.list_bucket_tree(bucket, prefix=prefix, recursive=True, token=HF_TOKEN):
        if isinstance(item, BucketFile):
            files[item.path] = int(item.size)
    return files


def verify_remote_exact(expected: dict[str, int], *, prefix: str, label: str) -> None:
    # Retry the *full* consistency check, not only the list request. This also
    # tolerates a short metadata-visibility delay after parallel S3 PUTs.
    def _check() -> None:
        actual = list_remote_files(DEST_BUCKET, prefix)
        missing = sorted(set(expected) - set(actual))
        extra = sorted(set(actual) - set(expected))
        mismatched = sorted(path for path in set(expected) & set(actual) if expected[path] != actual[path])
        if missing or extra or mismatched:
            raise RuntimeError(
                f"Remote inventory mismatch {label}: missing={missing[:5]}, "
                f"extra={extra[:5]}, size_mismatch={mismatched[:5]}"
            )

    retry(label, _check)


def verify_remote_paths(expected: dict[str, int], *, label: str) -> None:
    paths = list(expected)
    actual: dict[str, int] = {}
    for offset in range(0, len(paths), 500):
        chunk = paths[offset : offset + 500]
        items = retry(
            f"{label} paths {offset + 1}-{offset + len(chunk)}",
            lambda chunk=chunk: list(API.get_bucket_paths_info(DEST_BUCKET, chunk, token=HF_TOKEN)),
        )
        actual.update({item.path: int(item.size) for item in items})
    missing = sorted(set(expected) - set(actual))
    mismatched = sorted(path for path in set(expected) & set(actual) if expected[path] != actual[path])
    if missing or mismatched:
        raise RuntimeError(f"Remote verify lỗi {label}: missing={missing[:5]}, size={mismatched[:5]}")


def download_bucket_object(
    bucket: str,
    remote_path: str,
    local_path: Path,
    *,
    expected_size: int | None = None,
    expected_sha256: str | None = None,
) -> Path:
    local_path.parent.mkdir(parents=True, exist_ok=True)
    partial = local_path.with_name(local_path.name + ".part")
    if expected_size is None:
        remote_info = retry(
            f"stat {remote_path}",
            lambda: get_bucket_file(bucket, remote_path),
        )
        if remote_info is None:
            raise RuntimeError(f"Missing bucket object: {remote_path}")
        expected_size = int(remote_info.size)

    def _promote_if_complete(candidate: Path) -> bool:
        if not candidate.is_file():
            return False
        size = candidate.stat().st_size
        if expected_size is not None and size != int(expected_size):
            # A final path is never a valid partial. A .part smaller than the
            # target may still be resumable by the downloader, so retain it.
            if candidate == local_path or size > int(expected_size):
                remove_work_path(candidate)
            return False
        if expected_sha256 is not None and sha256_file(candidate) != expected_sha256:
            remove_work_path(candidate)
            return False
        if candidate != local_path:
            os.replace(candidate, local_path)
        return True

    # Handles a disconnect after all bytes were written but before rename.
    if _promote_if_complete(local_path) or _promote_if_complete(partial):
        return local_path

    def _download() -> None:
        API.download_bucket_files(
            bucket,
            files=[(remote_path, partial)],
            raise_on_missing_files=True,
            token=HF_TOKEN,
        )

    retry(f"download {remote_path}", _download)
    os.replace(partial, local_path)
    if expected_size is not None and local_path.stat().st_size != int(expected_size):
        remove_work_path(local_path)
        raise RuntimeError(f"Downloaded size mismatch: {remote_path}")
    if expected_sha256 is not None and sha256_file(local_path) != expected_sha256:
        remove_work_path(local_path)
        raise RuntimeError(f"Downloaded SHA-256 mismatch: {remote_path}")
    return local_path


def natural_video_key(video_id: str) -> tuple[str, int, int]:
    match = re.fullmatch(r"([A-Z]+)(\d+)_V(\d+)", video_id.upper())
    return (match.group(1), int(match.group(2)), int(match.group(3))) if match else (video_id, 0, 0)


print("Helpers loaded. Free disk: %.2f GiB" % (shutil.disk_usage(WORK_ROOT).free / GIB))


## Source manifests và metadata tối thiểu

Downloader chỉ tin `dataset_manifest.json`, shard manifest và metadata
`run_manifest/run_summary/map-keyframes`. Nó không dùng nested
`upload_state.corpus_audit.video_count` của uploader cũ (field đó từng ghi nhầm
category count). Metadata TAR được verify SHA-256, extract có chọn lọc, rồi xóa.

`local_tar` cho integrity mạnh nhất vì recompute SHA-256 toàn shard. `stream`
dùng HTTP Range để giảm disk peak; mode này bind checksum được khai báo, kiểm
exact remote byte size/range và toàn bộ member/map inventory, nhưng không
recompute được full TAR SHA-256 qua các request rời.


In [ ]:
SOURCE_MANIFEST_DIR = WORK_ROOT / "source_manifests"
SOURCE_MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

dataset_manifest_path = download_bucket_object(
    SOURCE_BUCKET,
    source_key("manifests/dataset_manifest.json"),
    SOURCE_MANIFEST_DIR / "dataset_manifest.json",
)
shard_manifest_path = download_bucket_object(
    SOURCE_BUCKET,
    source_key("manifests/keyframe_category_shards.jsonl"),
    SOURCE_MANIFEST_DIR / "keyframe_category_shards.jsonl",
)

SOURCE_DATASET_MANIFEST_BYTES = dataset_manifest_path.read_bytes()
SOURCE_DATASET_MANIFEST_SHA256 = sha256_bytes(SOURCE_DATASET_MANIFEST_BYTES)
SOURCE_DATASET_MANIFEST = json.loads(SOURCE_DATASET_MANIFEST_BYTES)
if int(SOURCE_DATASET_MANIFEST.get("videos", -1)) != EXPECTED_VIDEOS:
    raise RuntimeError("Source manifest không đủ 873 video")
if int(SOURCE_DATASET_MANIFEST.get("keyframes", -1)) != EXPECTED_KEYFRAMES:
    raise RuntimeError("Source manifest không đủ 1,339,055 keyframe")

SHARDS: dict[str, dict[str, Any]] = {}
with shard_manifest_path.open("r", encoding="utf-8") as stream:
    for line in stream:
        if not line.strip():
            continue
        row = json.loads(line)
        SHARDS[row["category"]] = row
if sorted(SHARDS) != ALL_CATEGORIES:
    raise RuntimeError(f"Shard categories sai: {sorted(SHARDS)}")

metadata_spec = SOURCE_DATASET_MANIFEST["metadata"]
metadata_remote = str(metadata_spec["remote_path"])
metadata_tar = WORK_ROOT / "downloads" / "infoshootpp_metadata.tar"
metadata_root = WORK_ROOT / "metadata"
metadata_marker = metadata_root / ".selected_extract.json"


def safe_selected_metadata_extract(archive_path: Path, destination: Path) -> None:
    selected_exact = {
        "infoshootpp/run_summary.csv",
        "infoshootpp/run_manifest.json",
        "infoshootpp/merge_manifest.json",
        "infoshootpp/metadata/postmerge_summary.json",
        "infoshootpp/metadata/postmerge_category_statistics.csv",
    }
    selected_prefixes = ["infoshootpp/map-keyframes/"]
    if CONFIG.get("extract_full_registry", False):
        selected_prefixes.append("infoshootpp/registry/")

    with tarfile.open(archive_path, mode="r:") as archive:
        for member in archive:
            name = member.name.rstrip("/")
            pure = PurePosixPath(name)
            if pure.is_absolute() or ".." in pure.parts:
                raise RuntimeError(f"Unsafe TAR member: {member.name}")
            selected = name in selected_exact or any(name.startswith(prefix) for prefix in selected_prefixes)
            if not selected:
                continue
            if member.issym() or member.islnk() or member.isdev():
                raise RuntimeError(f"Refuse special TAR member: {member.name}")
            target = (destination / pure).resolve()
            if destination.resolve() not in target.parents:
                raise RuntimeError(f"TAR traversal: {member.name}")
            if member.isdir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            if not member.isfile():
                raise RuntimeError(f"Unsupported TAR member: {member.name}")
            target.parent.mkdir(parents=True, exist_ok=True)
            source = archive.extractfile(member)
            if source is None:
                raise RuntimeError(f"Không đọc được TAR member: {member.name}")
            partial = target.with_name(target.name + ".part")
            with partial.open("wb") as output:
                shutil.copyfileobj(source, output, length=8 * 1024 * 1024)
            if partial.stat().st_size != member.size:
                raise RuntimeError(f"Metadata member size mismatch: {member.name}")
            os.replace(partial, target)


marker_ok = False
if metadata_marker.exists():
    marker = json.loads(metadata_marker.read_text(encoding="utf-8"))
    marker_ok = (
        marker.get("source_sha256") == metadata_spec["sha256"]
        and bool(marker.get("registry_extracted", False))
        == bool(CONFIG.get("extract_full_registry", False))
    )

if not marker_ok:
    if metadata_root.exists():
        for child in metadata_root.iterdir():
            if child.is_dir():
                shutil.rmtree(child)
            else:
                child.unlink()
    download_bucket_object(
        SOURCE_BUCKET,
        metadata_remote,
        metadata_tar,
        expected_size=int(metadata_spec["bytes"]),
        expected_sha256=str(metadata_spec["sha256"]),
    )
    if metadata_tar.stat().st_size != int(metadata_spec["bytes"]):
        raise RuntimeError("Metadata TAR byte size mismatch")
    # download_bucket_object already checked the full digest.
    metadata_sha = str(metadata_spec["sha256"])
    safe_selected_metadata_extract(metadata_tar, metadata_root)
    atomic_json(
        metadata_marker,
        {"source_sha256": metadata_sha, "extracted_at": utc_now(), "registry_extracted": bool(CONFIG.get("extract_full_registry", False))},
    )
    remove_work_path(metadata_tar)

INFO_ROOT = metadata_root / "infoshootpp"
MAP_ROOT = INFO_ROOT / "map-keyframes"
RUN_SUMMARY_PATH = INFO_ROOT / "run_summary.csv"
RUN_MANIFEST_PATH = INFO_ROOT / "run_manifest.json"
if not MAP_ROOT.is_dir() or not RUN_SUMMARY_PATH.is_file() or not RUN_MANIFEST_PATH.is_file():
    raise RuntimeError("Metadata extract thiếu map/run summary/run manifest")

print("Source dataset manifest SHA-256:", SOURCE_DATASET_MANIFEST_SHA256)
print("Metadata selected extract:", INFO_ROOT)
print("Metadata TAR local deleted:", not metadata_tar.exists())


In [ ]:
@dataclass(frozen=True)
class FrameMapRow:
    n: int
    pts_time: float
    fps: float
    frame_idx: int

    @property
    def frame_id_suffix(self) -> str:
        return f"f{self.frame_idx:08d}"

    @property
    def source_frame_name(self) -> str:
        return f"{self.frame_id_suffix}.jpg"

    @property
    def media_frame_name(self) -> str:
        # :03d is a minimum width. n=5152 remains "5152", never truncates.
        return f"{self.n:03d}.jpg"


def load_map_rows(path: Path) -> list[FrameMapRow]:
    rows: list[FrameMapRow] = []
    with path.open("r", encoding="utf-8-sig", newline="") as stream:
        reader = csv.DictReader(stream)
        if reader.fieldnames != ["n", "pts_time", "fps", "frame_idx"]:
            raise RuntimeError(f"Map header sai {path}: {reader.fieldnames}")
        previous_frame_idx = -1
        for expected_n, raw in enumerate(reader, 1):
            row = FrameMapRow(
                n=int(raw["n"]),
                pts_time=float(raw["pts_time"]),
                fps=float(raw["fps"]),
                frame_idx=int(raw["frame_idx"]),
            )
            if row.n != expected_n:
                raise RuntimeError(f"n không liên tục {path}: expected={expected_n}, got={row.n}")
            if row.frame_idx <= previous_frame_idx:
                raise RuntimeError(f"frame_idx không tăng nghiêm ngặt: {path}")
            if not math.isfinite(row.pts_time) or not math.isfinite(row.fps) or row.fps <= 0:
                raise RuntimeError(f"Map numeric invalid: {path}")
            # Deliberately do not reject tiny L25 pts_time regressions; frame_idx is canonical.
            rows.append(row)
            previous_frame_idx = row.frame_idx
    if not rows:
        raise RuntimeError(f"Map rỗng: {path}")
    return rows


with RUN_SUMMARY_PATH.open("r", encoding="utf-8-sig", newline="") as stream:
    RUN_SUMMARY_ROWS = list(csv.DictReader(stream))
RUN_SUMMARY_BY_VIDEO = {row["video_id"]: row for row in RUN_SUMMARY_ROWS}
if len(RUN_SUMMARY_ROWS) != EXPECTED_VIDEOS or len(RUN_SUMMARY_BY_VIDEO) != EXPECTED_VIDEOS:
    raise RuntimeError("run_summary không có đúng 873 video unique")

MAP_PATH_BY_VIDEO: dict[str, Path] = {}
CATEGORY_VIDEO_IDS: dict[str, list[str]] = {category: [] for category in ALL_CATEGORIES}
CATEGORY_FRAME_COUNTS: dict[str, int] = {category: 0 for category in ALL_CATEGORIES}

audit_bar = tqdm(total=EXPECTED_VIDEOS, desc="Audit 873 map CSV")
for category in ALL_CATEGORIES:
    for path in sorted((MAP_ROOT / category).glob("*.csv"), key=lambda item: natural_video_key(item.stem)):
        video_id = path.stem
        if video_id in MAP_PATH_BY_VIDEO:
            raise RuntimeError(f"Duplicate map: {video_id}")
        rows = load_map_rows(path)
        summary = RUN_SUMMARY_BY_VIDEO.get(video_id)
        if summary is None or int(summary["final_frames"]) != len(rows):
            raise RuntimeError(f"Map/run_summary mismatch: {video_id}")
        MAP_PATH_BY_VIDEO[video_id] = path
        CATEGORY_VIDEO_IDS[category].append(video_id)
        CATEGORY_FRAME_COUNTS[category] += len(rows)
        audit_bar.update(1)
audit_bar.close()

if len(MAP_PATH_BY_VIDEO) != EXPECTED_VIDEOS:
    raise RuntimeError(f"Expected 873 maps, got {len(MAP_PATH_BY_VIDEO)}")
if sum(CATEGORY_FRAME_COUNTS.values()) != EXPECTED_KEYFRAMES:
    raise RuntimeError("Map total không bằng 1,339,055")
if set(MAP_PATH_BY_VIDEO) != set(RUN_SUMMARY_BY_VIDEO):
    raise RuntimeError("Map IDs và run_summary IDs khác nhau")

for category in ALL_CATEGORIES:
    shard = SHARDS[category]
    if int(shard["video_count"]) != len(CATEGORY_VIDEO_IDS[category]):
        raise RuntimeError(f"Shard video count mismatch: {category}")
    if int(shard["frame_count"]) != CATEGORY_FRAME_COUNTS[category]:
        raise RuntimeError(f"Shard frame count mismatch: {category}")

print("Metadata audit PASS:", {"videos": len(MAP_PATH_BY_VIDEO), "keyframes": sum(CATEGORY_FRAME_COUNTS.values())})
print(pd.DataFrame([
    {"category": category, "videos": len(CATEGORY_VIDEO_IDS[category]), "keyframes": CATEGORY_FRAME_COUNTS[category], "tar_GiB": int(SHARDS[category]["tar_bytes"]) / GIB}
    for category in ALL_CATEGORIES
]).to_string(index=False))


## Identity projection giữ nguyên contract app

Với mỗi map row:

```text
source member   infoshootpp/keyframes/L21/L21_V001/f00000003.jpg
frame_id        L21_V001@f00000003        (canonical retrieval identity)
frame_idx       3
keyframe_n      2
submit id       L21/L21_V001/002           (compatibility identity)
destination     Keyframes/Keyframes_L21/L21_V001/002.jpg
```

Backend hiện tại lấy `keyframe_n` từ Milvus/Elastic và tự dựng destination
URL này. React/FastAPI không cần đổi code, nhưng **data index bắt buộc phải
reindex** theo final map trước khi đổi `MEDIA_BASE_URL`. Loader BTC hiện tại còn
reject 483 tiny `pts_time` regressions ở L25; migration index phải dùng
`frame_idx` làm canonical ordering hoặc cập nhật policy đó.


In [ ]:
def keyframe_media_key(category: str, video_id: str, n: int) -> str:
    return f"Keyframes/Keyframes_{category}/{video_id}/{n:03d}.jpg"


def video_media_key(video_id: str) -> str:
    group = video_id.split("_", 1)[0]
    return f"Videos/Videos_{group}/{video_id}.mp4"


def frame_id(video_id: str, frame_idx: int) -> str:
    return f"{video_id}@f{frame_idx:08d}"


def submit_keyframe_id(category: str, video_id: str, n: int) -> str:
    return f"{category}/{video_id}/{n:03d}"


def source_member_name(category: str, video_id: str, frame_idx: int) -> str:
    return f"infoshootpp/keyframes/{category}/{video_id}/f{frame_idx:08d}.jpg"


def commit_binding(category: str | None = None) -> dict[str, Any]:
    binding = {
        "pipeline_version": PIPELINE_VERSION,
        "layout_version": DEST_LAYOUT_VERSION,
        "source_bucket": SOURCE_BUCKET,
        "source_prefix": SOURCE_PREFIX,
        "source_dataset_manifest_sha256": SOURCE_DATASET_MANIFEST_SHA256,
        "dest_bucket": DEST_BUCKET,
        "dest_prefix": DEST_PREFIX,
    }
    if category is not None:
        binding.update(
            {
                "category": category,
                "source_tar_sha256": SHARDS[category]["sha256"],
                "source_tar_bytes": int(SHARDS[category]["tar_bytes"]),
            }
        )
    return binding


def validate_binding(value: dict[str, Any], expected: dict[str, Any], *, label: str) -> None:
    actual = value.get("binding")
    if actual != expected:
        raise RuntimeError(f"Resume binding mismatch {label}. Dùng destination prefix/bucket mới.")


def keyframe_progress_path(category: str) -> str:
    return dest_key(f"manifest/_migration/progress/keyframes/{category}.json")


def keyframe_video_commit_path(category: str, video_id: str) -> str:
    return dest_key(f"manifest/_migration/commits/keyframes/{category}/{video_id}.json")


def keyframe_category_commit_path(category: str) -> str:
    return dest_key(f"manifest/_migration/commits/keyframes/{category}.json")


def initial_keyframe_progress(category: str) -> dict[str, Any]:
    return {
        "schema": 1,
        "kind": "keyframe_category_progress",
        "binding": commit_binding(category),
        "status": "running",
        "videos": {},
        "stream_next_offset": 0,
        "updated_at": utc_now(),
    }


def load_keyframe_progress(category: str) -> dict[str, Any]:
    path = keyframe_progress_path(category)
    value = read_remote_json(DEST_BUCKET, path)
    if value is None:
        return initial_keyframe_progress(category)
    validate_binding(value, commit_binding(category), label=path)
    if not isinstance(value.get("videos"), dict):
        raise RuntimeError(f"Progress videos invalid: {path}")
    return value


def save_keyframe_progress(category: str, state: dict[str, Any]) -> None:
    state["updated_at"] = utc_now()
    upload_json(keyframe_progress_path(category), state, label=f"save {category} progress")


@dataclass
class VideoStage:
    category: str
    video_id: str
    mapping: list[FrameMapRow]
    mapping_by_idx: dict[int, FrameMapRow]
    materialize: bool
    local_dir: Path
    seen_sizes: dict[int, int]


def build_video_stage(category: str, video_id: str, *, materialize: bool) -> VideoStage:
    if video_id not in CATEGORY_VIDEO_IDS[category]:
        raise RuntimeError(f"Video ngoài metadata queue: {category}/{video_id}")
    mapping = load_map_rows(MAP_PATH_BY_VIDEO[video_id])
    local_dir = WORK_ROOT / "staging" / "keyframes" / category / video_id
    if local_dir.exists():
        remove_work_path(local_dir)
    if materialize:
        local_dir.mkdir(parents=True, exist_ok=True)
    mapping_by_idx = {row.frame_idx: row for row in mapping}
    if len(mapping_by_idx) != len(mapping):
        raise RuntimeError(f"Duplicate frame_idx trong map: {video_id}")
    return VideoStage(category, video_id, mapping, mapping_by_idx, materialize, local_dir, {})


def copy_tar_jpeg(archive: tarfile.TarFile, member: tarfile.TarInfo, destination: Path) -> None:
    source = archive.extractfile(member)
    if source is None:
        raise RuntimeError(f"Không đọc được member: {member.name}")
    partial = destination.with_name(destination.name + ".part")
    written = 0
    first = True
    with partial.open("wb") as output:
        while True:
            chunk = source.read(1024 * 1024)
            if not chunk:
                break
            if first:
                if not chunk.startswith(b"\xff\xd8"):
                    raise RuntimeError(f"Không phải JPEG: {member.name}")
                first = False
            output.write(chunk)
            written += len(chunk)
    if written != member.size:
        partial.unlink(missing_ok=True)
        raise RuntimeError(f"Extracted size mismatch: {member.name}")
    os.replace(partial, destination)


def consume_keyframe_member(stage: VideoStage, archive: tarfile.TarFile, member: tarfile.TarInfo, frame_idx: int) -> None:
    if not member.isfile() or member.issym() or member.islnk():
        raise RuntimeError(f"Expected regular JPEG: {member.name}")
    row = stage.mapping_by_idx.get(frame_idx)
    if row is None:
        raise RuntimeError(f"TAR frame không có trong map: {member.name}")
    expected_member = source_member_name(stage.category, stage.video_id, row.frame_idx)
    if member.name != expected_member:
        raise RuntimeError(f"Member identity mismatch: {member.name} != {expected_member}")
    if frame_idx in stage.seen_sizes:
        raise RuntimeError(f"Duplicate frame_idx trong TAR: {member.name}")
    stage.seen_sizes[frame_idx] = int(member.size)
    if stage.materialize:
        destination = stage.local_dir / row.media_frame_name
        copy_tar_jpeg(archive, member, destination)


def inventory_digest(stage: VideoStage) -> str:
    digest = hashlib.sha256()
    for row in stage.mapping:
        size = stage.seen_sizes[row.frame_idx]
        digest.update(
            f"{row.n}\t{row.frame_idx}\t{row.source_frame_name}\t{row.media_frame_name}\t{size}\n".encode()
        )
    return digest.hexdigest()


def valid_video_commit(commit: dict[str, Any], stage: VideoStage) -> bool:
    file_sizes = commit.get("file_sizes")
    return (
        commit.get("status") == "verified"
        and commit.get("binding") == commit_binding(stage.category)
        and commit.get("video_id") == stage.video_id
        and int(commit.get("frame_count", -1)) == len(stage.mapping)
        and isinstance(file_sizes, list)
        and len(file_sizes) == len(stage.mapping)
        and [int(size) for size in file_sizes]
        == [stage.seen_sizes[row.frame_idx] for row in stage.mapping]
        and int(commit.get("total_bytes", -1)) == sum(int(size) for size in file_sizes)
        and commit.get("inventory_sha256") == inventory_digest(stage)
    )


def finalize_keyframe_video(stage: VideoStage, state: dict[str, Any]) -> dict[str, Any]:
    expected_idx = [row.frame_idx for row in stage.mapping]
    if set(stage.seen_sizes) != set(expected_idx):
        missing = [idx for idx in expected_idx if idx not in stage.seen_sizes]
        extra = sorted(set(stage.seen_sizes) - set(expected_idx))
        raise RuntimeError(f"TAR/map mismatch {stage.video_id}: missing={missing[:5]}, extra={extra[:5]}")

    existing_summary = state["videos"].get(stage.video_id)
    commit_path = keyframe_video_commit_path(stage.category, stage.video_id)
    existing_commit = None
    if existing_summary is not None:
        existing_commit = read_remote_json(DEST_BUCKET, commit_path)
        if existing_commit is None or not valid_video_commit(existing_commit, stage):
            raise RuntimeError(f"Progress có video nhưng commit invalid: {stage.video_id}")
        if CONFIG["verify_completed_on_resume"]:
            expected = {
                dest_key(keyframe_media_key(stage.category, stage.video_id, row.n)): int(size)
                for row, size in zip(stage.mapping, existing_commit["file_sizes"])
            }
            verify_remote_exact(
                expected,
                prefix=dest_key(f"Keyframes/Keyframes_{stage.category}/{stage.video_id}") + "/",
                label=f"reverify {stage.video_id}",
            )
        if stage.local_dir.exists():
            remove_work_path(stage.local_dir)
        return existing_commit

    if not stage.materialize:
        existing_commit = read_remote_json(DEST_BUCKET, commit_path)
        if existing_commit is None or not valid_video_commit(existing_commit, stage):
            raise RuntimeError(f"Skip requested nhưng không có valid commit: {stage.video_id}")
        state["videos"][stage.video_id] = {
            "frame_count": int(existing_commit["frame_count"]),
            "total_bytes": int(existing_commit["total_bytes"]),
            "inventory_sha256": existing_commit["inventory_sha256"],
            "commit_path": commit_path,
        }
        save_keyframe_progress(stage.category, state)
        return existing_commit

    expected_remote: dict[str, int] = {}
    upload_items: list[tuple[Path, str]] = []
    file_sizes: list[int] = []
    for row in stage.mapping:
        local_file = stage.local_dir / row.media_frame_name
        size = int(stage.seen_sizes[row.frame_idx])
        if not local_file.is_file() or local_file.stat().st_size != size:
            raise RuntimeError(f"Local projection invalid: {local_file}")
        remote = dest_key(keyframe_media_key(stage.category, stage.video_id, row.n))
        expected_remote[remote] = size
        upload_items.append((local_file, remote))
        file_sizes.append(size)

    try:
        upload_keyframe_pairs(upload_items, label=f"upload keyframes {stage.video_id}")
        verify_remote_exact(
            expected_remote,
            prefix=dest_key(f"Keyframes/Keyframes_{stage.category}/{stage.video_id}") + "/",
            label=f"verify keyframes {stage.video_id}",
        )
        commit = {
            "schema": 1,
            "kind": "keyframe_video_commit",
            "status": "verified",
            "binding": commit_binding(stage.category),
            "video_id": stage.video_id,
            "category": stage.category,
            "frame_count": len(stage.mapping),
            "total_bytes": sum(file_sizes),
            "file_sizes": file_sizes,
            "inventory_sha256": inventory_digest(stage),
            "media_prefix": keyframe_media_key(stage.category, stage.video_id, 1).rsplit("/", 1)[0],
            "completed_at": utc_now(),
        }
        upload_json(commit_path, commit, label=f"commit keyframes {stage.video_id}")
        state["videos"][stage.video_id] = {
            "frame_count": len(stage.mapping),
            "total_bytes": sum(file_sizes),
            "inventory_sha256": commit["inventory_sha256"],
            "commit_path": commit_path,
        }
        save_keyframe_progress(stage.category, state)
        return commit
    finally:
        if stage.local_dir.exists():
            remove_work_path(stage.local_dir)


In [ ]:
def validate_tar_directory_member(member: tarfile.TarInfo, category: str) -> None:
    pure = PurePosixPath(member.name.rstrip("/"))
    if pure.is_absolute() or ".." in pure.parts:
        raise RuntimeError(f"Unsafe TAR member: {member.name}")
    if not member.isdir():
        raise RuntimeError(f"Unexpected non-JPEG member: {member.name}")
    allowed_roots = {
        "infoshootpp",
        "infoshootpp/keyframes",
        f"infoshootpp/keyframes/{category}",
    }
    value = pure.as_posix()
    if value not in allowed_roots and not re.fullmatch(
        rf"infoshootpp/keyframes/{re.escape(category)}/{re.escape(category)}_V\d+", value
    ):
        raise RuntimeError(f"Unexpected directory member: {member.name}")


def category_already_complete(category: str) -> bool:
    path = keyframe_category_commit_path(category)
    commit = read_remote_json(DEST_BUCKET, path)
    if commit is None:
        return False
    validate_binding(commit, commit_binding(category), label=path)
    expected_videos = CATEGORY_VIDEO_IDS[category]
    return (
        commit.get("status") == "verified"
        and int(commit.get("video_count", -1)) == len(expected_videos)
        and int(commit.get("frame_count", -1)) == CATEGORY_FRAME_COUNTS[category]
    )


def mark_category_complete(category: str, state: dict[str, Any], *, source_verification: str) -> None:
    expected_videos = CATEGORY_VIDEO_IDS[category]
    missing = [video_id for video_id in expected_videos if video_id not in state["videos"]]
    if missing:
        raise RuntimeError(f"Category {category} còn thiếu commit: {missing[:10]}")
    frame_count = sum(int(state["videos"][video_id]["frame_count"]) for video_id in expected_videos)
    total_bytes = sum(int(state["videos"][video_id]["total_bytes"]) for video_id in expected_videos)
    if frame_count != CATEGORY_FRAME_COUNTS[category]:
        raise RuntimeError(f"Category frame total mismatch: {category}")
    commit = {
        "schema": 1,
        "kind": "keyframe_category_commit",
        "status": "verified",
        "binding": commit_binding(category),
        "video_count": len(expected_videos),
        "frame_count": frame_count,
        "total_bytes": total_bytes,
        "source_verification": source_verification,
        "completed_at": utc_now(),
    }
    upload_json(keyframe_category_commit_path(category), commit, label=f"commit category {category}")
    state["status"] = "verified"
    state["stream_next_offset"] = None
    state["summary"] = {
        "video_count": len(expected_videos),
        "frame_count": frame_count,
        "total_bytes": total_bytes,
    }
    save_keyframe_progress(category, state)


def process_local_category_tar(category: str, tar_path: Path, state: dict[str, Any]) -> None:
    finished: set[str] = set()
    stage: VideoStage | None = None
    with tarfile.open(tar_path, mode="r:") as archive:
        for member in archive:
            match = MEMBER_RE.fullmatch(member.name)
            if match is None:
                validate_tar_directory_member(member, category)
                continue
            member_category = match.group("category")
            video_id = match.group("video_id")
            frame_idx_value = int(match.group("frame_idx"))
            if member_category != category:
                raise RuntimeError(f"Wrong category member in {category}.tar: {member.name}")
            if stage is None or stage.video_id != video_id:
                if stage is not None:
                    finalize_keyframe_video(stage, state)
                    finished.add(stage.video_id)
                if video_id in finished:
                    raise RuntimeError(f"Video members không contiguous: {video_id}")
                stage = build_video_stage(
                    category,
                    video_id,
                    materialize=video_id not in state["videos"],
                )
            consume_keyframe_member(stage, archive, member, frame_idx_value)
        if stage is not None:
            finalize_keyframe_video(stage, state)
            finished.add(stage.video_id)

    if finished != set(CATEGORY_VIDEO_IDS[category]):
        missing = sorted(set(CATEGORY_VIDEO_IDS[category]) - finished, key=natural_video_key)
        extra = sorted(finished - set(CATEGORY_VIDEO_IDS[category]), key=natural_video_key)
        raise RuntimeError(f"Category TAR video mismatch: missing={missing[:5]}, extra={extra[:5]}")


def open_source_tar_range(
    remote_path: str,
    start_offset: int,
    expected_size: int,
) -> requests.Response:
    headers = dict(REQUEST_HEADERS)
    headers["Accept-Encoding"] = "identity"
    if start_offset:
        headers["Range"] = f"bytes={start_offset}-"
    response = HTTP.get(
        resolve_url(SOURCE_BUCKET, remote_path),
        headers=headers,
        stream=True,
        allow_redirects=True,
        timeout=(int(CONFIG["http_connect_timeout"]), int(CONFIG["http_read_timeout"])),
    )
    expected_status = 206 if start_offset else 200
    if response.status_code != expected_status:
        response.close()
        raise RuntimeError(
            f"Range status {response.status_code}, expected {expected_status}, offset={start_offset}"
        )
    if start_offset:
        content_range = response.headers.get("Content-Range", "")
        match = re.fullmatch(r"bytes (\d+)-(\d+)/(\d+)", content_range)
        if (
            match is None
            or int(match.group(1)) != start_offset
            or int(match.group(3)) != expected_size
        ):
            response.close()
            raise RuntimeError(f"Unexpected Content-Range: {content_range}")
    elif response.headers.get("Content-Length") and int(response.headers["Content-Length"]) != expected_size:
        response.close()
        raise RuntimeError("Unexpected full TAR Content-Length")
    if response.headers.get("Content-Encoding", "identity").lower() not in {"", "identity"}:
        response.close()
        raise RuntimeError(f"TAR response is content-encoded: {response.headers.get('Content-Encoding')}")
    response.raw.decode_content = False
    return response


def stream_one_video(
    category: str,
    remote_path: str,
    start_offset: int,
    state: dict[str, Any],
) -> tuple[VideoStage | None, int | None]:
    response = open_source_tar_range(
        remote_path,
        start_offset,
        int(SHARDS[category]["tar_bytes"]),
    )
    stage: VideoStage | None = None
    try:
        with tarfile.open(fileobj=response.raw, mode="r|") as archive:
            for member in archive:
                match = MEMBER_RE.fullmatch(member.name)
                if match is None:
                    validate_tar_directory_member(member, category)
                    continue
                member_category = match.group("category")
                video_id = match.group("video_id")
                frame_idx_value = int(match.group("frame_idx"))
                if member_category != category:
                    raise RuntimeError(f"Wrong category member: {member.name}")
                if stage is None:
                    stage = build_video_stage(
                        category,
                        video_id,
                        materialize=video_id not in state["videos"],
                    )
                elif stage.video_id != video_id:
                    # TarInfo.offset is relative to this Range stream. Reopen exactly
                    # at this unconsumed next-video header after the current upload.
                    return stage, start_offset + int(member.offset)
                consume_keyframe_member(stage, archive, member, frame_idx_value)
        return stage, None
    finally:
        response.close()


def process_stream_category_tar(category: str, remote_path: str, state: dict[str, Any]) -> None:
    offset = int(state.get("stream_next_offset") or 0)
    processed_this_run: set[str] = set()
    while True:
        stage_root = WORK_ROOT / "staging" / "keyframes" / category
        if stage_root.exists():
            remove_work_path(stage_root)

        def _consume() -> tuple[VideoStage | None, int | None]:
            return stream_one_video(category, remote_path, offset, state)

        stage, next_offset = retry(f"stream {category} offset={offset}", _consume)
        if stage is None:
            break
        if stage.video_id in processed_this_run:
            raise RuntimeError(f"Stream loop repeated video: {stage.video_id}")
        finalize_keyframe_video(stage, state)
        processed_this_run.add(stage.video_id)
        if next_offset is None:
            break
        if next_offset <= offset:
            raise RuntimeError(f"Stream offset không tiến: {offset} -> {next_offset}")
        offset = next_offset
        state["stream_next_offset"] = offset
        save_keyframe_progress(category, state)


def choose_tar_mode(category: str) -> str:
    configured = CONFIG["tar_mode"]
    free = shutil.disk_usage(WORK_ROOT).free
    minimum_stage_free = int(
        (float(CONFIG["disk_reserve_gib"]) + float(CONFIG["max_video_stage_gib"])) * GIB
    )
    if free < minimum_stage_free:
        raise RuntimeError(
            f"Disk free {free / GIB:.2f} GiB không đủ reserve + one-video stage "
            f"{minimum_stage_free / GIB:.2f} GiB"
        )
    required = (
        int(SHARDS[category]["tar_bytes"])
        + int(float(CONFIG["disk_reserve_gib"]) * GIB)
        + int(float(CONFIG["max_video_stage_gib"]) * GIB)
    )
    if configured == "local_tar":
        if free < required:
            raise RuntimeError(
                f"Forced local_tar cần {required / GIB:.2f} GiB nhưng chỉ còn "
                f"{free / GIB:.2f} GiB; dùng tar_mode='auto' hoặc 'stream'"
            )
        return "local_tar"
    if configured == "stream":
        return "stream"
    return "local_tar" if free >= required else "stream"


def migrate_keyframe_category(category: str) -> None:
    if category_already_complete(category):
        print(f"[SKIP] {category}: verified category commit đã tồn tại")
        return
    state = load_keyframe_progress(category)
    shard = SHARDS[category]
    remote_path = str(shard["remote_path"])
    remote_info = get_bucket_file(SOURCE_BUCKET, remote_path)
    if remote_info is None or int(remote_info.size) != int(shard["tar_bytes"]):
        raise RuntimeError(f"Source remote TAR size mismatch: {category}")

    mode = choose_tar_mode(category)
    print(
        f"\n[{category}] mode={mode} | tar={int(shard['tar_bytes']) / GIB:.2f} GiB | "
        f"done={len(state['videos'])}/{len(CATEGORY_VIDEO_IDS[category])}"
    )
    local_tar = WORK_ROOT / "downloads" / f"{category}.tar"
    if mode == "local_tar":
        download_bucket_object(
            SOURCE_BUCKET,
            remote_path,
            local_tar,
            expected_size=int(shard["tar_bytes"]),
            expected_sha256=str(shard["sha256"]),
        )
        if local_tar.stat().st_size != int(shard["tar_bytes"]):
            remove_work_path(local_tar)
            raise RuntimeError(f"Downloaded TAR size mismatch: {category}")
        # download_bucket_object already checked the full digest.
        process_local_category_tar(category, local_tar, state)
        source_verification = "downloaded_size_and_sha256_plus_per_video_inventory"
    else:
        process_stream_category_tar(category, remote_path, state)
        source_verification = (
            "remote_size_plus_manifest_declared_sha256_plus_strict_member_and_per_video_inventory"
        )

    mark_category_complete(category, state, source_verification=source_verification)
    if local_tar.exists():
        remove_work_path(local_tar)
    print(f"[DONE] {category}; local TAR deleted={not local_tar.exists()}")


## Self-test cục bộ

Test không gọi network/write remote. Nó kiểm tra identity alias, parser CRLF,
`pts_time` regression policy và TAR traversal guard trước khi bắt đầu migration.


In [ ]:
def run_self_tests() -> None:
    assert keyframe_media_key("L21", "L21_V001", 2) == "Keyframes/Keyframes_L21/L21_V001/002.jpg"
    assert keyframe_media_key("L22", "L22_V016", 5152).endswith("/5152.jpg")
    assert frame_id("L21_V001", 3) == "L21_V001@f00000003"
    assert submit_keyframe_id("L21", "L21_V001", 2) == "L21/L21_V001/002"
    assert source_member_name("L21", "L21_V001", 3).endswith("/f00000003.jpg")

    test_map = WORK_ROOT / "selftest_map.csv"
    test_map.write_bytes(b"n,pts_time,fps,frame_idx\r\n1,0.1,25,3\r\n2,0.099,25,4\r\n")
    rows = load_map_rows(test_map)
    assert [row.n for row in rows] == [1, 2]
    assert [row.frame_idx for row in rows] == [3, 4]
    test_map.unlink()

    unsafe = tarfile.TarInfo("../escape")
    unsafe.type = tarfile.DIRTYPE
    try:
        validate_tar_directory_member(unsafe, "L21")
    except RuntimeError:
        pass
    else:
        raise AssertionError("Traversal member was not rejected")
    print("Self-tests PASS")


if CONFIG["run_self_tests"]:
    run_self_tests()


## Chạy keyframe migration

Cell này có thể chạy nhiều giờ/ngày. Log `[DONE] <video>` được commit remote
trước khi folder local bị xóa. Nếu runtime dừng, chạy lại từ đầu notebook;
category progress sẽ skip phần đã verified.


FAST S3 mode hiển thị `files/s` và `MiB/s` cho từng video. Với workload JPEG
nhỏ, hãy ưu tiên `files/s` khi đánh giá throughput. Nếu Colab/network ổn định,
`hf_s3_workers=64` là cấu hình khởi đầu aggressive; có thể thử 96 hoặc 128 nếu
CPU/RAM/network còn dư.


In [ ]:
if CONFIG["run_keyframes"]:
    for category in SELECTED_CATEGORIES:
        migrate_keyframe_category(category)
else:
    print("run_keyframes=False — skipped")


## Cloudflare R2 video → Colab → HF — VIDEO TURBO v4

Đây là coordinator cho **873 MP4** sau khi keyframe L21–L30 đã hoàn tất.

- mặc định `run_keyframes=False`, `run_videos=True`, `run_finalize=True`;
- R2 download chạy song song và SHA-256 được tính ngay trong luồng tải HTTP;
- benchmark bằng **video thật** để chọn Xet hoặc HF S3 theo **MiB/s**;
- download và upload overlap theo pipeline có `video_prefetch_limit`, không stage toàn corpus;
- commit/progress vẫn được ghi tuần tự để resume an toàn;
- nếu MP4 đã tồn tại đúng byte-size vì runtime chết trước commit, v4 chỉ tải lại source
  để khôi phục SHA-256 rồi **skip upload HF**;
- sau đủ 873 video, notebook tiếp tục final audit/manifests/`_SUCCESS.json`.

HF S3 gateway được cấu hình với multipart threshold/chunk 2 GiB theo khuyến nghị
của Hugging Face để tránh chia video lớn thành quá nhiều part. Native Xet dùng
adaptive concurrency thay vì pin cứng 64 stream như thử nghiệm JPEG trước đó.


In [ ]:
def video_progress_path() -> str:
    return dest_key("manifest/_migration/progress/videos.json")


def video_commit_path(video_id: str) -> str:
    category = video_id.split("_", 1)[0]
    return dest_key(f"manifest/_migration/commits/videos/{category}/{video_id}.json")


def initial_video_progress() -> dict[str, Any]:
    return {
        "schema": 1,
        "kind": "video_progress",
        "binding": commit_binding(),
        "source_mode": CONFIG["video_source_mode"],
        "source_identity": VIDEO_SOURCE_IDENTITY,
        "status": "running",
        "videos": {},
        "updated_at": utc_now(),
    }


def load_video_progress() -> dict[str, Any]:
    value = read_remote_json(DEST_BUCKET, video_progress_path())
    if value is None:
        return initial_video_progress()
    validate_binding(value, commit_binding(), label=video_progress_path())
    if (
        value.get("source_mode") != CONFIG["video_source_mode"]
        or value.get("source_identity") != VIDEO_SOURCE_IDENTITY
    ):
        raise RuntimeError("Video source identity đổi giữa chừng; dùng destination prefix mới")
    if not isinstance(value.get("videos"), dict):
        raise RuntimeError("Video progress videos invalid")
    return value


def save_video_progress(state: dict[str, Any]) -> None:
    state["updated_at"] = utc_now()
    upload_json(video_progress_path(), state, label="save video progress")


def build_r2_s3_client():
    account_id = read_secret("R2_ACCOUNT_ID")
    access_key = read_secret("R2_ACCESS_KEY_ID")
    secret_key = read_secret("R2_SECRET_ACCESS_KEY")
    bucket = read_secret("R2_BUCKET")
    client = boto3.client(
        "s3",
        endpoint_url=f"https://{account_id}.r2.cloudflarestorage.com",
        aws_access_key_id=access_key,
        aws_secret_access_key=secret_key,
        region_name="auto",
        config=BotoConfig(
            max_pool_connections=max(16, int(CONFIG["video_download_workers"]) * 4),
            connect_timeout=int(CONFIG["http_connect_timeout"]),
            read_timeout=int(CONFIG["http_read_timeout"]),
            tcp_keepalive=True,
            retries={"max_attempts": int(CONFIG["max_retries"]), "mode": "adaptive"},
        ),
    )
    source_identity = {
        "mode": "r2_s3",
        # Avoid writing the account id itself to public progress/commits.
        "account_bucket_sha256": sha256_bytes(f"{account_id}\n{bucket}".encode("utf-8")),
    }
    return client, bucket, source_identity


R2_S3 = None
R2_BUCKET_NAME = None
VIDEO_SOURCE_IDENTITY = None
if CONFIG["video_source_mode"] == "r2_s3":
    R2_S3, R2_BUCKET_NAME, VIDEO_SOURCE_IDENTITY = build_r2_s3_client()
elif CONFIG["video_source_mode"] == "public_http":
    VIDEO_SOURCE_IDENTITY = {
        "mode": "public_http",
        "base_url": str(CONFIG["r2_public_base_url"]).strip().rstrip("/"),
    }
elif CONFIG["video_source_mode"] != "public_http":
    raise ValueError("video_source_mode phải là public_http hoặc r2_s3")
assert VIDEO_SOURCE_IDENTITY is not None


_VIDEO_TLS = threading.local()


def _video_http() -> requests.Session:
    session = getattr(_VIDEO_TLS, "session", None)
    if session is None:
        session = requests.Session()
        adapter = requests.adapters.HTTPAdapter(
            pool_connections=4,
            pool_maxsize=4,
            max_retries=0,
        )
        session.mount("https://", adapter)
        session.mount("http://", adapter)
        _VIDEO_TLS.session = session
    return session


def public_r2_url(key: str) -> str:
    return f"{str(CONFIG['r2_public_base_url']).rstrip('/')}/{quote(key, safe='/')}"


def public_http_head(key: str) -> dict[str, Any]:
    response = _video_http().head(
        public_r2_url(key),
        allow_redirects=True,
        timeout=(int(CONFIG["http_connect_timeout"]), int(CONFIG["http_read_timeout"])),
    )
    try:
        if response.status_code != 200:
            raise RuntimeError(f"R2 HEAD {response.status_code}: {key}")
        size = response.headers.get("Content-Length")
        if not size:
            raise RuntimeError(f"R2 HEAD thiếu Content-Length: {key}")
        return {
            "size": int(size),
            "etag": response.headers.get("ETag"),
            "last_modified": response.headers.get("Last-Modified"),
            "source_url": public_r2_url(key),
        }
    finally:
        response.close()


def _hash_existing_prefix(path: Path, size: int) -> Any:
    digest = hashlib.sha256()
    if size <= 0:
        return digest
    with path.open("rb") as stream:
        remaining = size
        while remaining:
            chunk = stream.read(min(16 * 1024 * 1024, remaining))
            if not chunk:
                raise RuntimeError(f"Partial file ngắn hơn expected prefix: {path}")
            digest.update(chunk)
            remaining -= len(chunk)
    return digest


def download_public_http_hashed(key: str, destination: Path, expected_size: int) -> tuple[str, float]:
    """Resume-capable R2 download; SHA-256 is computed during the network pass."""
    partial = destination.with_name(destination.name + ".part")
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and destination.stat().st_size == expected_size:
        started = time.perf_counter()
        digest = sha256_file(destination)
        return digest, time.perf_counter() - started
    if destination.exists():
        destination.unlink()
    if partial.exists() and partial.stat().st_size > expected_size:
        partial.unlink()

    started_total = time.perf_counter()

    def _attempt() -> str:
        existing = partial.stat().st_size if partial.exists() else 0
        if existing > expected_size:
            partial.unlink(missing_ok=True)
            existing = 0
        if existing == expected_size and expected_size > 0:
            return sha256_file(partial)

        headers = {"Accept-Encoding": "identity"}
        if existing:
            headers["Range"] = f"bytes={existing}-"

        response = _video_http().get(
            public_r2_url(key),
            headers=headers,
            stream=True,
            allow_redirects=True,
            timeout=(int(CONFIG["http_connect_timeout"]), int(CONFIG["http_read_timeout"])),
        )

        # If Range is ignored, restart from zero using this full 200 response.
        if existing and response.status_code == 200:
            partial.unlink(missing_ok=True)
            existing = 0
        elif existing and response.status_code != 206:
            response.close()
            raise RuntimeError(f"R2 resume status={response.status_code}: {key}")
        elif not existing and response.status_code != 200:
            response.close()
            raise RuntimeError(f"R2 download status={response.status_code}: {key}")

        if existing:
            content_range = response.headers.get("Content-Range", "")
            match = re.fullmatch(r"bytes (\d+)-(\d+)/(\d+)", content_range)
            if (
                match is None
                or int(match.group(1)) != existing
                or int(match.group(3)) != expected_size
            ):
                response.close()
                raise RuntimeError(f"R2 Content-Range invalid {content_range!r}: {key}")

        digest = _hash_existing_prefix(partial, existing)
        mode = "ab" if existing else "wb"
        try:
            with partial.open(mode) as output:
                for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
                    if chunk:
                        output.write(chunk)
                        digest.update(chunk)
        finally:
            response.close()

        if partial.stat().st_size != expected_size:
            if partial.stat().st_size > expected_size:
                partial.unlink(missing_ok=True)
            raise RuntimeError(
                f"R2 partial size mismatch {partial.stat().st_size if partial.exists() else 0} "
                f"!= {expected_size}: {key}"
            )
        return digest.hexdigest()

    digest = retry(f"download R2 {key}", _attempt)
    os.replace(partial, destination)
    return digest, time.perf_counter() - started_total


def s3_head(key: str) -> dict[str, Any]:
    assert R2_S3 is not None and R2_BUCKET_NAME is not None
    result = R2_S3.head_object(Bucket=R2_BUCKET_NAME, Key=key)
    return {
        "size": int(result["ContentLength"]),
        "etag": result.get("ETag"),
        "last_modified": result.get("LastModified").isoformat() if result.get("LastModified") else None,
        "source_url": f"s3://{R2_BUCKET_NAME}/{key}",
    }


def download_s3_hashed(key: str, destination: Path, expected_size: int) -> tuple[str, float]:
    """R2 S3 fallback. boto3 downloads efficiently; digest is one local pass."""
    assert R2_S3 is not None and R2_BUCKET_NAME is not None
    partial = destination.with_name(destination.name + ".part")
    destination.parent.mkdir(parents=True, exist_ok=True)
    started = time.perf_counter()
    retry(
        f"download R2 S3 {key}",
        lambda: R2_S3.download_file(R2_BUCKET_NAME, key, str(partial)),
    )
    if partial.stat().st_size != expected_size:
        partial.unlink(missing_ok=True)
        raise RuntimeError(f"R2 S3 size mismatch: {key}")
    os.replace(partial, destination)
    digest = sha256_file(destination)
    return digest, time.perf_counter() - started


def video_source_head(key: str) -> dict[str, Any]:
    if CONFIG["video_source_mode"] == "public_http":
        return retry(f"HEAD R2 {key}", lambda: public_http_head(key))
    return retry(f"HEAD R2 S3 {key}", lambda: s3_head(key))


def video_source_download(
    key: str,
    destination: Path,
    expected_size: int,
) -> tuple[str, float]:
    if CONFIG["video_source_mode"] == "public_http":
        return download_public_http_hashed(key, destination, expected_size)
    return download_s3_hashed(key, destination, expected_size)


@dataclass
class PreparedVideo:
    video_id: str
    media_key: str
    remote_path: str
    source: dict[str, Any]
    local_dir: Path
    local_file: Path
    sha256: str
    download_seconds: float
    remote_already_complete: bool


@dataclass
class UploadedVideo:
    prepared: PreparedVideo
    backend: str
    upload_seconds: float
    upload_mib_per_s: float


def _remote_video_size_matches(remote_path: str, expected_size: int) -> bool:
    if not CONFIG.get("video_resume_remote_size_match", True):
        return False
    info = retry(
        f"probe remote video {remote_path}",
        lambda: get_bucket_file(DEST_BUCKET, remote_path),
    )
    return info is not None and int(info.size) == int(expected_size)


def prepare_video(video_id: str) -> PreparedVideo:
    media_key = video_media_key(video_id)
    remote_path = dest_key(media_key)
    source = video_source_head(media_key)
    size = int(source["size"])

    free = shutil.disk_usage(WORK_ROOT).free
    required = size + int(float(CONFIG["disk_reserve_gib"]) * GIB)
    if free < required:
        raise RuntimeError(
            f"Disk free {free / GIB:.2f} GiB không đủ stage {video_id} "
            f"({size / GIB:.2f} GiB) + reserve"
        )

    local_dir = WORK_ROOT / "staging" / "videos" / video_id
    local_dir.mkdir(parents=True, exist_ok=True)
    local_file = local_dir / f"{video_id}.mp4"

    remote_complete = _remote_video_size_matches(remote_path, size)
    digest, seconds = video_source_download(media_key, local_file, size)
    mib_s = size / 2**20 / max(seconds, 1e-9)
    print(
        f"[VIDEO DL] {video_id}: {size / 2**20:.1f} MiB in {seconds:.1f}s "
        f"| {mib_s:.1f} MiB/s"
        + (" | HF media already present; upload will be skipped" if remote_complete else ""),
        flush=True,
    )
    return PreparedVideo(
        video_id=video_id,
        media_key=media_key,
        remote_path=remote_path,
        source=source,
        local_dir=local_dir,
        local_file=local_file,
        sha256=digest,
        download_seconds=seconds,
        remote_already_complete=remote_complete,
    )


_VIDEO_S3_TRANSFER_CONFIG = TransferConfig(
    multipart_threshold=max(5 * 1024 * 1024, int(float(CONFIG["video_s3_multipart_threshold_gib"]) * GIB)),
    multipart_chunksize=max(5 * 1024 * 1024, int(float(CONFIG["video_s3_multipart_chunksize_gib"]) * GIB)),
    max_concurrency=max(1, int(CONFIG["video_s3_part_workers"])),
    use_threads=True,
)


def _upload_video_xet(prepared: PreparedVideo) -> UploadedVideo:
    size = int(prepared.source["size"])
    if prepared.remote_already_complete:
        return UploadedVideo(prepared, "EXISTING", 0.0, float("inf"))

    started = time.perf_counter()
    retry(
        f"upload video {prepared.video_id} XET",
        lambda: batch_bucket_files(
            DEST_BUCKET,
            add=[(prepared.local_file, prepared.remote_path)],
            token=HF_TOKEN,
        ),
    )
    seconds = time.perf_counter() - started
    verify_remote_paths(
        {prepared.remote_path: size},
        label=f"verify video {prepared.video_id}",
    )
    mib_s = size / 2**20 / max(seconds, 1e-9)
    print(
        f"[VIDEO UP] {prepared.video_id}: {size / 2**20:.1f} MiB in {seconds:.1f}s "
        f"| {mib_s:.1f} MiB/s | backend=XET",
        flush=True,
    )
    return UploadedVideo(prepared, "XET", seconds, mib_s)


def _upload_video_s3(prepared: PreparedVideo) -> UploadedVideo:
    if HF_S3 is None:
        raise RuntimeError("HF S3 client chưa được cấu hình")
    size = int(prepared.source["size"])
    if prepared.remote_already_complete:
        return UploadedVideo(prepared, "EXISTING", 0.0, float("inf"))

    started = time.perf_counter()

    def _upload() -> None:
        HF_S3.upload_file(
            str(prepared.local_file),
            DEST_S3_BUCKET,
            prepared.remote_path,
            ExtraArgs={"ContentType": "video/mp4"},
            Config=_VIDEO_S3_TRANSFER_CONFIG,
        )

    retry(f"upload video {prepared.video_id} S3", _upload, max_retries=4)
    seconds = time.perf_counter() - started
    verify_remote_paths(
        {prepared.remote_path: size},
        label=f"verify video {prepared.video_id}",
    )
    mib_s = size / 2**20 / max(seconds, 1e-9)
    print(
        f"[VIDEO UP] {prepared.video_id}: {size / 2**20:.1f} MiB in {seconds:.1f}s "
        f"| {mib_s:.1f} MiB/s | backend=S3",
        flush=True,
    )
    return UploadedVideo(prepared, "S3", seconds, mib_s)


def _commit_uploaded_video(uploaded: UploadedVideo, state: dict[str, Any]) -> None:
    prepared = uploaded.prepared
    source = prepared.source
    size = int(source["size"])

    # For an interrupted prior runtime we may have skipped upload because the
    # exact-size MP4 already exists. Verify it before committing.
    if uploaded.backend == "EXISTING":
        verify_remote_paths(
            {prepared.remote_path: size},
            label=f"verify existing video {prepared.video_id}",
        )

    commit = {
        "schema": 1,
        "kind": "video_commit",
        "status": "verified",
        "binding": commit_binding(),
        "video_id": prepared.video_id,
        "category": prepared.video_id.split("_", 1)[0],
        "source_mode": CONFIG["video_source_mode"],
        "source_identity": VIDEO_SOURCE_IDENTITY,
        "source_key": prepared.media_key,
        "source_url": source["source_url"],
        "source_etag": source.get("etag"),
        "source_last_modified": source.get("last_modified"),
        "media_key": prepared.media_key,
        "hf_path": prepared.remote_path,
        "size_bytes": size,
        "sha256": prepared.sha256,
        "upload_backend": uploaded.backend,
        "download_seconds": round(float(prepared.download_seconds), 6),
        "upload_seconds": round(float(uploaded.upload_seconds), 6),
        "completed_at": utc_now(),
    }
    commit_path = video_commit_path(prepared.video_id)
    upload_json(commit_path, commit, label=f"commit video {prepared.video_id}")
    state["videos"][prepared.video_id] = {
        "size_bytes": size,
        "sha256": prepared.sha256,
        "commit_path": commit_path,
    }
    save_video_progress(state)
    print(
        f"[DONE video] {prepared.video_id} | {size / GIB:.2f} GiB "
        f"| upload={uploaded.backend}",
        flush=True,
    )
    if prepared.local_dir.exists():
        remove_work_path(prepared.local_dir)


def _cleanup_prepared(prepared: PreparedVideo) -> None:
    if prepared.local_dir.exists():
        remove_work_path(prepared.local_dir)


def _benchmark_video_backend(
    pending_ids: list[str],
    state: dict[str, Any],
) -> tuple[str, list[str]]:
    """Benchmark Xet vs S3 on real videos. Every benchmark upload is committed."""
    if VIDEO_UPLOAD_BACKEND in {"xet", "s3"}:
        return VIDEO_UPLOAD_BACKEND, pending_ids
    if HF_S3 is None:
        return "xet", pending_ids

    per_backend = max(1, int(CONFIG["video_benchmark_videos_per_backend"]))
    need = per_backend * 2
    bench_ids = pending_ids[:need]
    rest = pending_ids[need:]
    if len(bench_ids) < 2:
        return "xet", pending_ids

    print(
        f"[VIDEO TURBO] benchmark XET vs S3 using {len(bench_ids)} real MP4(s)...",
        flush=True,
    )

    prepared_list: list[PreparedVideo] = []
    with ThreadPoolExecutor(
        max_workers=min(int(CONFIG["video_download_workers"]), len(bench_ids)),
        thread_name_prefix="video-bench-dl",
    ) as pool:
        futures = {pool.submit(prepare_video, video_id): video_id for video_id in bench_ids}
        for future in as_completed(futures):
            prepared_list.append(future.result())

    # Preserve corpus order for deterministic assignment.
    order = {video_id: i for i, video_id in enumerate(bench_ids)}
    prepared_list.sort(key=lambda item: order[item.video_id])

    groups = {
        "xet": prepared_list[:per_backend],
        "s3": prepared_list[per_backend:],
    }
    scores: dict[str, list[tuple[int, float]]] = {"xet": [], "s3": []}

    try:
        for backend in ("xet", "s3"):
            for prepared in groups[backend]:
                if prepared.remote_already_complete:
                    uploaded = UploadedVideo(prepared, "EXISTING", 0.0, float("inf"))
                    _commit_uploaded_video(uploaded, state)
                    continue
                uploaded = (
                    _upload_video_xet(prepared)
                    if backend == "xet"
                    else _upload_video_s3(prepared)
                )
                scores[backend].append((int(prepared.source["size"]), uploaded.upload_seconds))
                _commit_uploaded_video(uploaded, state)
    except Exception:
        for prepared in prepared_list:
            if prepared.video_id not in state["videos"]:
                _cleanup_prepared(prepared)
        raise

    def aggregate_mib_s(values: list[tuple[int, float]]) -> float:
        if not values:
            return 0.0
        return sum(size for size, _ in values) / 2**20 / max(sum(sec for _, sec in values), 1e-9)

    xet_score = aggregate_mib_s(scores["xet"])
    s3_score = aggregate_mib_s(scores["s3"])
    chosen = "xet" if xet_score >= s3_score else "s3"
    print(
        "[VIDEO TURBO] benchmark results:\n"
        f"  XET: {xet_score:.1f} MiB/s\n"
        f"  S3 : {s3_score:.1f} MiB/s\n"
        f"[VIDEO TURBO] SELECTED: {chosen.upper()}",
        flush=True,
    )
    return chosen, rest


def _run_video_pipeline(
    pending_ids: list[str],
    backend: str,
    state: dict[str, Any],
) -> None:
    """Bounded download→upload pipeline with serial durable commits."""
    if not pending_ids:
        return

    download_workers = max(1, int(CONFIG["video_download_workers"]))
    upload_workers = max(
        1,
        int(
            CONFIG["video_xet_upload_workers"]
            if backend == "xet"
            else CONFIG["video_s3_upload_workers"]
        ),
    )
    prefetch_limit = max(upload_workers, int(CONFIG["video_prefetch_limit"]))

    pending = deque(pending_ids)
    ready: deque[PreparedVideo] = deque()
    download_futures: dict[Any, str] = {}
    upload_futures: dict[Any, PreparedVideo] = {}

    print(
        f"[VIDEO TURBO] pipeline backend={backend.upper()} | "
        f"download_workers={download_workers} | upload_workers={upload_workers} | "
        f"prefetch_limit={prefetch_limit}",
        flush=True,
    )

    def staged_count() -> int:
        return len(download_futures) + len(ready) + len(upload_futures)

    with ThreadPoolExecutor(
        max_workers=download_workers,
        thread_name_prefix="video-dl",
    ) as download_pool, ThreadPoolExecutor(
        max_workers=upload_workers,
        thread_name_prefix=f"video-{backend}",
    ) as upload_pool:
        try:
            while pending or download_futures or ready or upload_futures:
                made_progress = False

                # Keep only a bounded number of local MP4s in flight/staged.
                while pending and staged_count() < prefetch_limit:
                    video_id = pending.popleft()
                    download_futures[download_pool.submit(prepare_video, video_id)] = video_id
                    made_progress = True

                # Harvest finished downloads without blocking.
                for future in list(download_futures):
                    if not future.done():
                        continue
                    video_id = download_futures.pop(future)
                    try:
                        ready.append(future.result())
                    except Exception as exc:
                        raise RuntimeError(f"Prepare video failed: {video_id}") from exc
                    made_progress = True

                # Feed upload stage.
                while ready and len(upload_futures) < upload_workers:
                    prepared = ready.popleft()
                    fn = _upload_video_xet if backend == "xet" else _upload_video_s3
                    upload_futures[upload_pool.submit(fn, prepared)] = prepared
                    made_progress = True

                # Durable commit happens only on the coordinator thread, avoiding
                # races on the shared videos.json progress object.
                for future in list(upload_futures):
                    if not future.done():
                        continue
                    prepared = upload_futures.pop(future)
                    try:
                        uploaded = future.result()
                        _commit_uploaded_video(uploaded, state)
                    except Exception:
                        _cleanup_prepared(prepared)
                        raise
                    made_progress = True

                if not made_progress:
                    time.sleep(0.15)
        except Exception:
            # Best-effort cleanup of completed/staged local files. Running worker
            # futures are allowed to finish as executors unwind.
            for prepared in list(ready) + list(upload_futures.values()):
                _cleanup_prepared(prepared)
            raise


def migrate_all_videos() -> None:
    state = load_video_progress()
    video_ids = sorted(MAP_PATH_BY_VIDEO, key=natural_video_key)

    completed = set(state["videos"])
    pending_ids = [video_id for video_id in video_ids if video_id not in completed]
    print(
        f"[VIDEO TURBO] progress: done={len(completed)}/{len(video_ids)}, "
        f"pending={len(pending_ids)}",
        flush=True,
    )

    backend, pending_after_bench = _benchmark_video_backend(pending_ids, state)
    _run_video_pipeline(pending_after_bench, backend, state)

    missing = [video_id for video_id in video_ids if video_id not in state["videos"]]
    if missing:
        raise RuntimeError(f"Video progress missing: {missing[:10]}")

    state["status"] = "verified"
    state["summary"] = {
        "video_count": len(video_ids),
        "total_bytes": sum(
            int(state["videos"][video_id]["size_bytes"])
            for video_id in video_ids
        ),
        "video_turbo_version": "v4",
    }
    save_video_progress(state)
    print("Video migration PASS:", state["summary"])


if CONFIG["run_videos"]:
    migrate_all_videos()
else:
    print("run_videos=False — skipped")


## Final audit và publish manifests

Finalizer yêu cầu đủ commit của cả 10 category và đúng 873 video, bất kể chúng
được tạo bởi runtime hiện tại hay các Colab trước. Nó sinh:

```text
manifest/media_manifest.csv
manifest/video_manifest.csv
manifest/keyframe_map.jsonl        # exact Elastic keyframe-map documents
manifest/keyframes/L21.parquet ... L30.parquet
manifest/media_manifest.schema.json
manifest/dataset_manifest.json
manifest/_SUCCESS.json             # upload cuối cùng
```

Bảy cột đầu của CSV giữ schema R2 cũ. `frame_idx` nay có nghĩa đúng là decode
frame index; `keyframe_n` là ordinal riêng. Frontend không cần tải global CSV;
backend tiếp tục trả URL trực tiếp.


In [ ]:
MANIFEST_COLUMNS = [
    # Legacy R2-compatible prefix columns.
    "kind", "group", "video_id", "frame_name", "frame_idx", "source_path", "r2_key",
    # Correct extended identity/media contract.
    "category", "category_hint", "submit_category", "keyframe_n", "keyframe_name",
    "keyframe_id", "submit_keyframe_id", "image_id", "frame_id", "pts_time", "fps",
    "duration_sec", "source_frame_name", "source_tar", "source_member", "source_etag",
    "media_key", "hf_path", "hf_bucket_uri", "public_url", "size_bytes", "sha256",
    "migration_version",
]

MANIFEST_ARROW_SCHEMA = pa.schema(
    [
        pa.field("kind", pa.string(), nullable=False),
        pa.field("group", pa.string(), nullable=False),
        pa.field("video_id", pa.string(), nullable=False),
        pa.field("frame_name", pa.string()),
        pa.field("frame_idx", pa.int64()),
        pa.field("source_path", pa.string()),
        pa.field("r2_key", pa.string(), nullable=False),
        pa.field("category", pa.string(), nullable=False),
        pa.field("category_hint", pa.string()),
        pa.field("submit_category", pa.string()),
        pa.field("keyframe_n", pa.int64()),
        pa.field("keyframe_name", pa.string()),
        pa.field("keyframe_id", pa.string()),
        pa.field("submit_keyframe_id", pa.string()),
        pa.field("image_id", pa.string()),
        pa.field("frame_id", pa.string()),
        pa.field("pts_time", pa.float64()),
        pa.field("fps", pa.float32()),
        pa.field("duration_sec", pa.float64()),
        pa.field("source_frame_name", pa.string()),
        pa.field("source_tar", pa.string()),
        pa.field("source_member", pa.string()),
        pa.field("source_etag", pa.string()),
        pa.field("media_key", pa.string(), nullable=False),
        pa.field("hf_path", pa.string(), nullable=False),
        pa.field("hf_bucket_uri", pa.string(), nullable=False),
        pa.field("public_url", pa.string(), nullable=False),
        pa.field("size_bytes", pa.int64(), nullable=False),
        pa.field("sha256", pa.string()),
        pa.field("migration_version", pa.string(), nullable=False),
    ]
)


def load_all_commits() -> tuple[dict[str, dict[str, Any]], dict[str, dict[str, Any]]]:
    keyframe_commits: dict[str, dict[str, Any]] = {}
    for category in ALL_CATEGORIES:
        category_commit = read_remote_json(DEST_BUCKET, keyframe_category_commit_path(category))
        if category_commit is None:
            raise RuntimeError(f"Thiếu category commit: {category}")
        validate_binding(category_commit, commit_binding(category), label=f"category {category}")
        if category_commit.get("status") != "verified":
            raise RuntimeError(f"Category chưa verified: {category}")
        for video_id in CATEGORY_VIDEO_IDS[category]:
            commit = read_remote_json(DEST_BUCKET, keyframe_video_commit_path(category, video_id))
            if commit is None:
                raise RuntimeError(f"Thiếu keyframe video commit: {video_id}")
            expected_count = len(load_map_rows(MAP_PATH_BY_VIDEO[video_id]))
            if (
                commit.get("binding") != commit_binding(category)
                or commit.get("status") != "verified"
                or int(commit.get("frame_count", -1)) != expected_count
                or len(commit.get("file_sizes", [])) != expected_count
            ):
                raise RuntimeError(f"Invalid keyframe commit: {video_id}")
            keyframe_commits[video_id] = commit

    video_state = load_video_progress()
    if video_state.get("status") != "verified":
        raise RuntimeError("Video migration chưa verified")
    video_commits: dict[str, dict[str, Any]] = {}
    for video_id in sorted(MAP_PATH_BY_VIDEO, key=natural_video_key):
        commit = read_remote_json(DEST_BUCKET, video_commit_path(video_id))
        if (
            commit is None
            or commit.get("status") != "verified"
            or commit.get("binding") != commit_binding()
            or commit.get("source_identity") != VIDEO_SOURCE_IDENTITY
        ):
            raise RuntimeError(f"Invalid video commit: {video_id}")
        video_commits[video_id] = commit
    return keyframe_commits, video_commits


def audit_all_remote_media(
    keyframe_commits: dict[str, dict[str, Any]],
    video_commits: dict[str, dict[str, Any]],
) -> dict[str, int]:
    """Re-list every final media object before allowing a success marker."""

    keyframe_count = 0
    keyframe_bytes = 0
    for category in ALL_CATEGORIES:
        category_prefix = dest_key(f"Keyframes/Keyframes_{category}") + "/"
        expected_videos = set(CATEGORY_VIDEO_IDS[category])

        def _audit_category() -> tuple[int, int]:
            seen_per_video = {video_id: 0 for video_id in expected_videos}
            count = 0
            total_bytes = 0
            for item in API.list_bucket_tree(
                DEST_BUCKET,
                prefix=category_prefix,
                recursive=True,
                token=HF_TOKEN,
            ):
                if not isinstance(item, BucketFile):
                    continue
                relative = item.path[len(category_prefix) :]
                parts = relative.split("/")
                if len(parts) != 2:
                    raise RuntimeError(f"Unexpected keyframe path: {item.path}")
                video_id, filename = parts
                if video_id not in expected_videos or not filename.endswith(".jpg"):
                    raise RuntimeError(f"Unexpected keyframe object: {item.path}")
                stem = filename[:-4]
                if not stem.isdigit():
                    raise RuntimeError(f"Invalid served keyframe name: {item.path}")
                n = int(stem)
                file_sizes = keyframe_commits[video_id]["file_sizes"]
                if n < 1 or n > len(file_sizes) or filename != f"{n:03d}.jpg":
                    raise RuntimeError(f"Out-of-contract keyframe name: {item.path}")
                expected_size = int(file_sizes[n - 1])
                if int(item.size) != expected_size:
                    raise RuntimeError(f"Remote keyframe size mismatch: {item.path}")
                seen_per_video[video_id] += 1
                count += 1
                total_bytes += int(item.size)

            wrong_counts = {
                video_id: (seen_per_video[video_id], len(keyframe_commits[video_id]["file_sizes"]))
                for video_id in expected_videos
                if seen_per_video[video_id] != len(keyframe_commits[video_id]["file_sizes"])
            }
            if wrong_counts:
                raise RuntimeError(f"Remote keyframe count mismatch {category}: {list(wrong_counts.items())[:5]}")
            return count, total_bytes

        category_count, category_bytes = retry(
            f"final remote audit {category}",
            _audit_category,
        )
        if category_count != CATEGORY_FRAME_COUNTS[category]:
            raise RuntimeError(f"Final remote audit count mismatch: {category}")
        keyframe_count += category_count
        keyframe_bytes += category_bytes
        print(f"[AUDIT] {category}: {category_count:,} JPEG | {category_bytes / GIB:.2f} GiB")

    expected_videos = {
        dest_key(video_media_key(video_id)): int(commit["size_bytes"])
        for video_id, commit in video_commits.items()
    }
    verify_remote_paths(expected_videos, label="final remote audit videos")
    if keyframe_count != EXPECTED_KEYFRAMES or len(expected_videos) != EXPECTED_VIDEOS:
        raise RuntimeError("Final remote media totals mismatch")
    return {
        "keyframes": keyframe_count,
        "keyframe_bytes": keyframe_bytes,
        "videos": len(expected_videos),
        "video_bytes": sum(expected_videos.values()),
    }


def keyframe_manifest_row(
    category: str,
    video_id: str,
    row: FrameMapRow,
    size_bytes: int,
) -> dict[str, Any]:
    media_key = keyframe_media_key(category, video_id, row.n)
    hf_path = dest_key(media_key)
    source_member = source_member_name(category, video_id, row.frame_idx)
    return {
        "kind": "keyframe",
        "group": category,
        "video_id": video_id,
        "frame_name": row.media_frame_name,
        "frame_idx": row.frame_idx,
        "source_path": source_member,
        "r2_key": media_key,
        "category": category,
        "category_hint": category,
        "submit_category": category,
        "keyframe_n": row.n,
        "keyframe_name": f"{row.n:03d}",
        "keyframe_id": f"{video_id}/{row.n:03d}",
        "submit_keyframe_id": submit_keyframe_id(category, video_id, row.n),
        "image_id": submit_keyframe_id(category, video_id, row.n),
        "frame_id": frame_id(video_id, row.frame_idx),
        "pts_time": row.pts_time,
        "fps": row.fps,
        "duration_sec": None,
        "source_frame_name": row.source_frame_name,
        "source_tar": str(SHARDS[category]["remote_path"]),
        "source_member": source_member,
        "source_etag": None,
        "media_key": media_key,
        "hf_path": hf_path,
        "hf_bucket_uri": f"hf://buckets/{DEST_BUCKET}/{hf_path}",
        "public_url": f"{MEDIA_BASE_URL}/{quote(media_key, safe='/')}",
        "size_bytes": int(size_bytes),
        "sha256": None,
        "migration_version": PIPELINE_VERSION,
    }


def video_manifest_row(video_id: str, commit: dict[str, Any]) -> dict[str, Any]:
    summary = RUN_SUMMARY_BY_VIDEO[video_id]
    category = video_id.split("_", 1)[0]
    media_key = video_media_key(video_id)
    hf_path = dest_key(media_key)
    return {
        "kind": "video",
        "group": category,
        "video_id": video_id,
        "frame_name": None,
        "frame_idx": None,
        "source_path": commit["source_key"],
        "r2_key": media_key,
        "category": category,
        "category_hint": None,
        "submit_category": None,
        "keyframe_n": None,
        "keyframe_name": None,
        "keyframe_id": None,
        "submit_keyframe_id": None,
        "image_id": None,
        "frame_id": None,
        "pts_time": None,
        "fps": float(summary["fps"]),
        "duration_sec": float(summary["duration_sec"]),
        "source_frame_name": None,
        "source_tar": None,
        "source_member": None,
        "source_etag": commit.get("source_etag"),
        "media_key": media_key,
        "hf_path": hf_path,
        "hf_bucket_uri": f"hf://buckets/{DEST_BUCKET}/{hf_path}",
        "public_url": f"{MEDIA_BASE_URL}/{quote(media_key, safe='/')}",
        "size_bytes": int(commit["size_bytes"]),
        "sha256": commit["sha256"],
        "migration_version": PIPELINE_VERSION,
    }


def elastic_keyframe_record(manifest_row: dict[str, Any]) -> dict[str, Any]:
    # Exact field contract consumed by elastic_upload.py. Generating this
    # directly avoids the legacy KeyframeIndex pts_time-monotonicity guard.
    return {
        "keyframe_id": manifest_row["keyframe_id"],
        "submit_keyframe_id": manifest_row["submit_keyframe_id"],
        "video_id": manifest_row["video_id"],
        "category_hint": manifest_row["category_hint"],
        "submit_category": manifest_row["submit_category"],
        "keyframe_n": manifest_row["keyframe_n"],
        "keyframe_name": manifest_row["keyframe_name"],
        "pts_time": manifest_row["pts_time"],
        "fps": manifest_row["fps"],
        "frame_idx": manifest_row["frame_idx"],
    }


def write_parquet_rows(path: Path, rows: Iterable[dict[str, Any]], *, batch_rows: int) -> int:
    path.parent.mkdir(parents=True, exist_ok=True)
    writer = pq.ParquetWriter(path, MANIFEST_ARROW_SCHEMA, compression="zstd")
    count = 0
    batch: list[dict[str, Any]] = []
    try:
        for row in rows:
            batch.append(row)
            if len(batch) >= batch_rows:
                writer.write_table(pa.Table.from_pylist(batch, schema=MANIFEST_ARROW_SCHEMA))
                count += len(batch)
                batch.clear()
        if batch:
            writer.write_table(pa.Table.from_pylist(batch, schema=MANIFEST_ARROW_SCHEMA))
            count += len(batch)
    finally:
        writer.close()
    return count


def publish_manifests() -> dict[str, Any]:
    keyframe_commits, video_commits = load_all_commits()
    remote_media_audit = audit_all_remote_media(keyframe_commits, video_commits)
    out_dir = WORK_ROOT / "manifests" / "publish"
    if out_dir.exists():
        remove_work_path(out_dir)
    out_dir.mkdir(parents=True)

    media_csv = out_dir / "media_manifest.csv"
    video_csv = out_dir / "video_manifest.csv"
    keyframe_map_jsonl = out_dir / "keyframe_map.jsonl"
    schema_json = out_dir / "media_manifest.schema.json"
    upload_files: list[tuple[Path, str]] = []
    keyframe_rows_written = 0

    csv_handle = media_csv.open("w", encoding="utf-8", newline="") if CONFIG["write_compat_media_manifest_csv"] else None
    csv_writer = csv.DictWriter(csv_handle, fieldnames=MANIFEST_COLUMNS) if csv_handle else None
    keyframe_map_handle = keyframe_map_jsonl.open("w", encoding="utf-8")
    if csv_writer:
        csv_writer.writeheader()

    try:
        for category in ALL_CATEGORIES:
            parquet_path = out_dir / "keyframes" / f"{category}.parquet"

            def iter_category_rows(category=category):
                nonlocal keyframe_rows_written
                for video_id in CATEGORY_VIDEO_IDS[category]:
                    mapping = load_map_rows(MAP_PATH_BY_VIDEO[video_id])
                    sizes = keyframe_commits[video_id]["file_sizes"]
                    for map_row, size in zip(mapping, sizes):
                        manifest_row = keyframe_manifest_row(category, video_id, map_row, int(size))
                        if csv_writer:
                            csv_writer.writerow(manifest_row)
                        keyframe_map_handle.write(
                            json.dumps(
                                elastic_keyframe_record(manifest_row),
                                ensure_ascii=False,
                                separators=(",", ":"),
                            )
                            + "\n"
                        )
                        keyframe_rows_written += 1
                        yield manifest_row

            count = write_parquet_rows(
                parquet_path,
                iter_category_rows(),
                batch_rows=int(CONFIG["manifest_parquet_batch_rows"]),
            )
            if count != CATEGORY_FRAME_COUNTS[category]:
                raise RuntimeError(f"Parquet row count mismatch: {category}")
            upload_files.append((parquet_path, dest_key(f"manifest/keyframes/{category}.parquet")))

        with video_csv.open("w", encoding="utf-8", newline="") as output:
            writer = csv.DictWriter(output, fieldnames=MANIFEST_COLUMNS)
            writer.writeheader()
            for video_id in sorted(video_commits, key=natural_video_key):
                row = video_manifest_row(video_id, video_commits[video_id])
                writer.writerow(row)
                if csv_writer:
                    csv_writer.writerow(row)
    finally:
        if csv_handle:
            csv_handle.close()
        keyframe_map_handle.close()

    if keyframe_rows_written != EXPECTED_KEYFRAMES:
        raise RuntimeError(f"Global keyframe manifest rows={keyframe_rows_written}")

    schema_payload = {
        "schema_version": 1,
        "columns": MANIFEST_COLUMNS,
        "identity_contract": {
            "canonical_frame_id": "{video_id}@f{frame_idx:08d}",
            "elastic_keyframe_id": "{video_id}/{keyframe_n:03d}",
            "compat_submit_keyframe_id": "{category}/{video_id}/{keyframe_n:03d}",
            "image_id": "same as submit_keyframe_id",
            "source_jpeg": "f{frame_idx:08d}.jpg",
            "served_jpeg": "{keyframe_n:03d}.jpg",
            "warning": "keyframe_n and frame_idx are distinct; never infer one from the other",
        },
        "media_base_url": MEDIA_BASE_URL,
        "cutover_requirement": (
            "Reindex to final InfoShot++ keyframe_n and ensure every category returned by retrieval "
            "exists in this destination before changing the single global MEDIA_BASE_URL"
        ),
    }
    atomic_json(schema_json, schema_payload)

    upload_files.append((video_csv, dest_key("manifest/video_manifest.csv")))
    upload_files.append((keyframe_map_jsonl, dest_key("manifest/keyframe_map.jsonl")))
    upload_files.append((schema_json, dest_key("manifest/media_manifest.schema.json")))
    if media_csv.exists():
        upload_files.append((media_csv, dest_key("manifest/media_manifest.csv")))

    expected_remote = {remote: local.stat().st_size for local, remote in upload_files}
    upload_pairs(upload_files, label="upload final manifest data")
    verify_remote_paths(expected_remote, label="verify final manifest data")

    file_records = [
        {
            "path": remote,
            "bytes": local.stat().st_size,
            "sha256": sha256_file(local),
        }
        for local, remote in upload_files
    ]
    dataset_manifest = {
        "schema": 1,
        "format": "aic26-hf-media-v1",
        "status": "complete",
        "pipeline_version": PIPELINE_VERSION,
        "layout_version": DEST_LAYOUT_VERSION,
        "created_at": utc_now(),
        "source": {
            "bucket": SOURCE_BUCKET,
            "prefix": SOURCE_PREFIX,
            "dataset_manifest_sha256": SOURCE_DATASET_MANIFEST_SHA256,
            "videos": EXPECTED_VIDEOS,
            "keyframes": EXPECTED_KEYFRAMES,
        },
        "destination": {
            "bucket": DEST_BUCKET,
            "prefix": DEST_PREFIX,
            "media_base_url": MEDIA_BASE_URL,
            "public": True,
        },
        "cutover": {
            "media_ready": True,
            "destination_category_scope": ALL_CATEGORIES,
            "retrieval_index_migrated_by_this_notebook": False,
            "requirements": [
                (
                    "Reindex Milvus/Elasticsearch to final InfoShot++ keyframe_n; "
                    "join canonical identity by video_id + frame_idx"
                ),
                (
                    "Ensure live retrieval only returns L21-L30, or mirror every legacy category "
                    "such as K01-K20 into the same destination before changing MEDIA_BASE_URL"
                ),
            ],
        },
        "counts": {
            "videos": EXPECTED_VIDEOS,
            "keyframes": EXPECTED_KEYFRAMES,
            "categories": 10,
            "video_bytes": remote_media_audit["video_bytes"],
            "keyframe_bytes": remote_media_audit["keyframe_bytes"],
        },
        "category_videos": {category: len(CATEGORY_VIDEO_IDS[category]) for category in ALL_CATEGORIES},
        "category_keyframes": CATEGORY_FRAME_COUNTS,
        "remote_media_audit": {
            **remote_media_audit,
            "method": "recursive exact path/count/size audit for JPEG; exact path/size audit for MP4",
            "audited_at": utc_now(),
        },
        "files": file_records,
    }
    dataset_manifest_remote = dest_key("manifest/dataset_manifest.json")
    dataset_size, dataset_sha = upload_json(
        dataset_manifest_remote,
        dataset_manifest,
        label="publish dataset_manifest.json",
    )

    success = {
        "schema": 1,
        "complete": True,
        "pipeline_version": PIPELINE_VERSION,
        "layout_version": DEST_LAYOUT_VERSION,
        "completed_at": utc_now(),
        "videos": EXPECTED_VIDEOS,
        "keyframes": EXPECTED_KEYFRAMES,
        "media_base_url": MEDIA_BASE_URL,
        "dataset_manifest": dataset_manifest_remote,
        "dataset_manifest_bytes": dataset_size,
        "dataset_manifest_sha256": dataset_sha,
        "retrieval_index_migrated_by_this_notebook": False,
        "cutover_warning": (
            "Do not change MEDIA_BASE_URL while indexes still emit BTC keyframe_n or any category "
            "whose media is absent from this L21-L30 destination"
        ),
        "examples": {
            "keyframe": f"{MEDIA_BASE_URL}/Keyframes/Keyframes_L21/L21_V001/001.jpg",
            "video": f"{MEDIA_BASE_URL}/Videos/Videos_L21/L21_V001.mp4",
        },
    }
    # The next cell performs public browser checks, then publishes this marker
    # as the absolute final remote write.
    return success


FINAL_SUCCESS = None
SUCCESS_PUBLISHED = False
if CONFIG["run_finalize"]:
    FINAL_SUCCESS = publish_manifests()
    print("Manifest data verified; public browser checks and _SUCCESS are still pending.")
else:
    print("run_finalize=False — skipped")


## Browser/CORS/Range checks

Frontend dùng `<img>`, `<video crossOrigin="anonymous">`, canvas capture và seek.
Vì vậy public JPEG phải trả CORS, còn MP4 phải hỗ trợ byte Range. Cell kiểm tra
không gửi HF token, đúng như browser production.


In [ ]:
def browser_checks() -> dict[str, Any]:
    image_url = f"{MEDIA_BASE_URL}/Keyframes/Keyframes_L21/L21_V001/001.jpg"
    video_url = f"{MEDIA_BASE_URL}/Videos/Videos_L21/L21_V001.mp4"
    origin = "https://aic26-frontend.example"

    image_response = HTTP.get(
        image_url,
        headers={"Origin": origin, "Range": "bytes=0-1023"},
        stream=True,
        allow_redirects=True,
        timeout=(int(CONFIG["http_connect_timeout"]), int(CONFIG["http_read_timeout"])),
    )
    video_response = HTTP.get(
        video_url,
        headers={"Origin": origin, "Range": "bytes=0-1023"},
        stream=True,
        allow_redirects=True,
        timeout=(int(CONFIG["http_connect_timeout"]), int(CONFIG["http_read_timeout"])),
    )
    result = {
        "image_status": image_response.status_code,
        "image_content_type": image_response.headers.get("Content-Type"),
        "image_cors": image_response.headers.get("Access-Control-Allow-Origin"),
        "video_status": video_response.status_code,
        "video_content_type": video_response.headers.get("Content-Type"),
        "video_content_range": video_response.headers.get("Content-Range"),
        "video_accept_ranges": video_response.headers.get("Accept-Ranges"),
        "video_cors": video_response.headers.get("Access-Control-Allow-Origin"),
    }
    image_response.close()
    video_response.close()
    if CONFIG["strict_browser_checks"]:
        if result["image_status"] not in {200, 206} or not str(result["image_content_type"]).startswith("image/jpeg"):
            raise RuntimeError(f"JPEG browser check fail: {result}")
        if result["video_status"] != 206 or not result["video_content_range"]:
            raise RuntimeError(f"MP4 Range check fail: {result}")
        if result["image_cors"] not in {"*", origin} or result["video_cors"] not in {"*", origin}:
            raise RuntimeError(f"CORS check fail: {result}")
    return result


if FINAL_SUCCESS is not None:
    BROWSER_CHECKS = browser_checks()
    FINAL_SUCCESS["browser_checks"] = BROWSER_CHECKS
    # Publish success LAST. Consumers must ignore a migration without this marker.
    upload_json(
        dest_key("manifest/_SUCCESS.json"),
        FINAL_SUCCESS,
        label="publish _SUCCESS.json LAST",
    )
    SUCCESS_PUBLISHED = True
    print(json.dumps(FINAL_SUCCESS, ensure_ascii=False, indent=2))
else:
    print("Finalization chưa chạy; browser checks skipped")


## Cleanup cuối runtime

Remote commits/manifests là resume source of truth. Sau khi mọi cell công việc
được chọn trong runtime này chạy xong, toàn bộ `work_root` local được xóa để trả
disk Colab (kể cả worker có `run_finalize=False`). Không có lệnh delete remote.


In [ ]:
if CONFIG["cleanup_work_root_at_end"]:
    if WORK_ROOT.exists():
        shutil.rmtree(WORK_ROOT)
    print("Local work root deleted:", not WORK_ROOT.exists())

if SUCCESS_PUBLISHED:
    print("\nMIGRATION COMPLETE")
    print("MEDIA_BASE_URL=" + MEDIA_BASE_URL)
    print("Success marker:", f"hf://buckets/{DEST_BUCKET}/{dest_key('manifest/_SUCCESS.json')}")

if SUCCESS_PUBLISHED and CONFIG["disconnect_runtime_after_success"]:
    from google.colab import runtime
    runtime.unassign()
